LLM PIPELINE FINAL

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import json
import PyPDF2

Prompts

In [ ]:
import json

def generate_system_prompt(paper, task, dataset_names):
    title = paper['title']
    content = paper['text']

    # Extract title first and then reuse in other tasks
    if task == "title":
        return f"""
        You are tasked with extracting the title of the provided cybersecurity paper.
        
        Rules:
        \t1. The title is often at the top of the first page.
        \t2. Extract the title in its entirety.
        
        Here is the paper content:
        <Start of Paper Content>
        {content}
        <End of Paper Content>
        
        Your response must be returned in the following JSON format:
        {{
            "title": "Title of the paper here"
        }}

        Your response: """
    

    elif task == "dataset_name":
        datasets = [
            {
                "dataset_name": "Name of the first dataset you find",
                "action": "created or used",
                "doi": "DOI for the first dataset you find. If not available, this should be an empty string.",
                "url": "URL link to the first dataset you find. If not available, this should be an empty string.",
            },
            {
                "dataset_name": "Name of the second dataset you find",
                "action": "created or used",
                "doi": "DOI for the second dataset you find. If not available, this should be an empty string.",
                "url": "URL link to the second dataset you find. If not available, this should be an empty string.",
            }
        ]
        datasets_json = json.dumps({"datasets": datasets}, indent=4)
        
        return f"""
        You are tasked with identifying and extracting datasets from the cybersecurity paper titled "{title}".

        IMPORTANT: This task is ONLY for dataset extraction. DO NOT assign categories here.
        
        Guidelines and Rules:
        **STRICTLY FOLLOW ALL GUIDELINES**
        \t1. Definition: A dataset is a named collection of data (e.g., raw data such as network traffic, survey responses,
         and audio files records, logs, traces, binaries, as well as processed data such as annotated website links and qualitative codebooks or observations) used for training, testing, evaluation, or analysis.
        \t2. Dataset Action:
        \t. A dataset is considered **used** if it is an existing dataset that is directly used within the study (e.g., for training, evaluation, benchmarking, comparison, or analysis), regardless of whether it originates from public repositories or prior research papers where it was introduced as a contribution.
        \t. A dataset is considered **created** if it is explicitly introduced by the authors as a contribution, exists beyond the body of the paper, or created for training, testing, or evaluating.
        \t- Datasets can be mentioned explicitly by name (e.g., "We use UNSW-NB15") or implicitly (e.g., "we use the dataset from [25]" 
        if reference [25] clearly points to a dataset).


        \t3. **Be Comprehensive & Systematic:**
        \t- If multiple mentions refer to the same underlying dataset but differ in naming, description, or statistics (e.g., number of samples, users, records, or time span), they must be merged into a single dataset entry and not treated as separate datasets.
        If the same dataset is mentioned multiple times under slightly different names or descriptions (e.g., naming variations, statistics 
        such as number of users, samples, or time ranges), consider them as referring to the same dataset and merge them into a single entry. 
        (e.g., In the paper “BFId: Identity Inference Attacks Utilizing Beamforming Feedback Information” a single WiFi sensing dataset containing
        BFI and CSI recordings from 197 participants).
   
        \t. **Real Example from a Paper**:
        Consider the ACM CCS paper "Recompose Event Sequences vs. Predict Next Events: A Novel Anomaly Detection Approach for Discrete Event Logs"
        as an example:
        \t- Introduction: "DabLog achieves 97.18% and 80.25% F1 scores in evaluation upon HDFS system logs and UNSW-NB15 traffic logs..."
        \t- Motivation section: "Both methods were evaluated upon the same HDFS dataset [38, 39]..."
        \t- Evaluation section: "We evaluate DabLog with two datasets: UNSW-NB15 traffic logs [29] and HDFS console logs [39]..."
       
        \t4. DO NOT confuse datasets with the following (these are NOT datasets):
        \t. **Benchmark Suites / Evaluation Frameworks**:
              Examples: SPEC CPU2006, SPEC CPU2007, SPEC CPU2017  
              These are performance benchmarking suites, not datasets.  
              Even if the paper says:
              - “we used SPEC CPU2017”
              - “we evaluated on SPEC CPU2006”
              → DO NOT extract them as datasets.
          
        \t. **Software / Applications / Systems**:
              Examples: NGINX, PostgreSQL  
              These are systems or applications, not datasets.
        \t. **Libraries / Packages / Tools / Frameworks**:
              Examples: OpenSSL, libc.so  
              These are software components, not datasets.
              
        \t. **Operating System Components / Kernel Modules**:
              Examples: ipv6.ko, nf_tables.ko  
              These are system components, not datasets.
              
        \t. **Network Protocols**:
              Examples: TCP, UDP, IPv4, IPv6  
              These are communication protocols, not datasets.
              
        \t. **Models / Methods / Algorithms**:
              Examples: GPT-4, BERT, ResNet  
              These are models, not datasets.
              
        \t5. IMPORTANT DISAMBIGUATION RULE:
        A term is NOT a dataset simply because:
        - it is “used”
        - it appears in experiments 
        - it is evaluated or benchmarked
        It must be explicitly described as a **collection of data**.
        

        \t6. If the paper discusses a benchmark suite, tool, protocol, attack, or vulnerability but does not explicitly introduce a named data collection, return no dataset for that mention.
      
        \t7. For each dataset, identify:
        \t- **Name** of the dataset.
        \t- **action**: created or used
        \t- **DOI**: the DOI of the dataset (if available)
        \t- **URL**: the URL link of the dataset (if available)

         \t8- Do NOT hallucinate datasets

        \t9. For every dataset you extract, you MUST also assign a dataset category.
        If a dataset is identified, its category CANNOT be null or empty. Do NOT return a dataset without a category.

        
        \t- Example: In the "A Lightweight IoT Cryptojacking Detection Mechanism in Heterogeneous Smart Home Networks", you should return output like:
        "datasets": [
            {{
                "dataset_name": "IoT Cryptojacking Dataset",
                "action": "created",
                "doi": "",
                "url": "https://github.com/cslfiu/IoTCryptojacking",
            }},
            {{
                "dataset_name": "Benign Dataset",
                "action": "used",
                "doi": "http://doi.org/10.17632/5pmnkshffm.1",
                "url": "",
            }}
           ]
         }}

    
        \t\t- (e.g., in the paper “(Un)informed Consent: Studying GDPR Consent Notices in the Field”, the authors created a 
        large-scale dataset of 82,890 website visitors’ interaction logs and survey responses, thats falls under "User Activities" subdomain;
        although different subsets (e.g., experiments, survey participants) are used in later analysis, clickstream data is not seperate dataset here, they are one author just used different names everywhere, 
        they all originate from the same data collection and must be treated as a single dataset,
        regardless of how they are referenced as “used” in different sections). Return following output:

         "datasets": [
            {{
                "dataset_name": "consent notice interaction dataset",
                "action": "created",
                "doi": "",
                "url": "",
            }}
           ]
         }}

        \t10. If a dataset is extracted, its category cannot be null or empty.



        Here is the paper:
        <Start of Paper Content>
        {content}
        <End of Paper Content>

        Your output must be returned in valid JSON format:

        {datasets_json}
    
        Your response: """
    
    elif task == "dataset_categories":
        
        return f"""

        You are tasked with identifying the specific categories and subcategories of datasets extracted from the **dataset_name** task used in the cybersecurity paper titled "{title}".

        Clarification:
        Focus on the dataset's inherent characteristics and contents.
        Note: These categories are derived from the taxonomy outlined in the USENIX paper "Cybersecurity Research Datasets: Taxonomy and Empirical Analysis" by 
        Zheng et al., which provides a structured framework for categorizing cybersecurity datasets. Additionally, a new category for multimedia 
        data has been added based on evolving research needs.


        Rules:

        \t1. By **dataset_categories**, we mean identifying whether a dataset belongs to the following major categories and
their subcategories. If a dataset does not clearly fit any existing category, you may introduce a new category or subcategory only when strongly supported by evidence from the paper. Do not force a dataset into an incorrect category.

        **Major Categories and Subcategories**:

        \t\t- **Attacker-Related**:
        \t\t  1. **Attacks**: contain information on attempts to harm digital assets perpetrated intentionally by malicious actors. (e.g., extraction of 29K rental scam postings from Craigslist website).
        \t\t  2. **Vulnerabilities**: contain information on weaknesses in digital assets that can be exploited by an attacker. (e.g., CVE databases or Open Source Vulnerability Database).
        \t\t\t- Rule: Synthetic patterns or code constructs designed to test tools should not be classified as vulnerabilities unless they represent real-world weaknesses.
        \t\t  3. **Exploits**: contain information on how attacks may be perpetrated, but not when a particular system has been targeted by a malicious actor. (e.g., exploits from Microsoft security advisories)
        \t\t  4. **Cybercrime Infrastructures**: describe unlawful activities distinct from attacks, as well as
        information on the infrastructure and operations used by malicious actors to perpetrate attacks. (A typical example
        is the crawl of the Silk Road anonymous marketplace). Or more examples can be data coming from CrimeBB underground forum, Nulled database, and Dark Net Markets (DNM), Although the dataset contains user-generated content, it originates from underground marketplaces and forums 
        that constitute part of the cybercrime ecosystem so classify them as "Cybercrime Infrastructures".
        \t\t  5. **Malware**: is a curated collection of data samples that contain malicious software (malware) or artifacts derived from it. Raw binaries or executables (e.g., .exe, .apk, .elf files).

        \t\t- **Defender Artifacts**:
        \t\t  1. **Alerts**:  contain outputs of defender artifacts, such as firewall logs or blackhole traffic.
        \t\t  2. **Configurations**: contain information about how defender artifacts are set up and configured (e.g., SSL certificate configurations).
        \t\t  3. **Logs**: Raw defender-generated telemetry capturing system or network events without prior detection or alert processing.
        \t\t  Example: CTDD consists of system logs capturing both normal and malicious activity, and is therefore classified as Defender Logs rather than Attacks.
        \t\t- **User & Organizational Characteristics**:
        \t\t  1. **User Activities**:contain information about users or organizations online behavior, such as tweets (e.g., large-scale experiments capturing how users accept or reject GDPR cookie consent notices).
        \t\t  2. **User Attributes**: contain information about the characteristics of users or organizations themselves (e.g., user profiles). Example: If a dataset captures biometric signals (e.g., eye movements, keystrokes, gait for security analysis) and is used for authentication or identification, classify it as **User Attributes**.
        \t\t  3. **User Attitudes**: contain information about opinions or attitudes towards an issue, often gleaned through surveys.

        \t\t- **Macro-Level Internet Characteristics**:
        \t\t  1. **Applications**: contain information about Internet end products and services such as websites, Android apps, bitcoin, extensions, or code. A typical example of Applications is the Alexa list of top websites.
        \t\t\t- (e.g., open-source PHP web applications from GitHub and Sourcecodester, grouped into GL, GM, GH, and SC, used in “Testability Tarpits: the Impact of Code Patterns”, classified as **Applications** since they represent software systems rather than explicit vulnerability records).
        \t\t  2. **Network Traces**: are usually network traffic dumps that not only contain information regarding the application level, but also information about lower layers. Data usually comes from a benign resource, like an organization’s internal network, but malicious traffic might be included. For example, packet-level traces for Tor Pluggable Transport traffic collected in controlled environments.
        \t\t\t Example: (e.g., normal (benign) network traffic such as web browsing, video streaming, and file downloads collected from IoT devices in a smart home network).
        \t\t  3. **Topology**: datasets contain information about relationships between Internet components. A typical example in this category is CAIDA’s AS relationship database.
        \t\t  4. **Benchmarks**: contain information about measurements of Internet performance, such as upload/download speed or end-to-end network reliability. For example in the paper "Tackling bufferbloat in 3G/4G networks"  Jiang and Wang constructed a dataset that measured 3G/4G network performance in the US and Korea.
        \t\t  5. **Adverse Events**: contain information on events that harm digital assets where malicious intent has not been established (e.g., outages caused by routing misconfigurations).

        \t\t\t- Rule: **Dont confuse between these**: Network Traces contain raw traffic (packets/flows) regardless of benign or malicious content, while Attacks contain explicit malicious content (e.g., phishing, exploits), Defender Artifacts are system/security-generated logs or alerts, and Malware consists of malicious software or its artifacts.
        (e.g., IoT web + cryptojacking traffic → **Network Traces**; phishing emails → **Attacks**; firewall logs → **Defender Artifacts**; malware binaries/C2 traces → **Malware**).
        \t\t- **General-Purpose Data Modalities**:
        \t\t- 1. **Vision-Based Datasets**: Datasets originally developed for image or video tasks (e.g., CIFAR-10, MNIST, UCF101, Kinetics).
        \t\t- 2. **Audio/Speech Datasets**: Datasets containing audio data used for tasks such as spoofing detection or authentication (e.g., SpeechCommands, LibriSpeech).
        \t\t- 3. **Sensor/Behavioral Datasets**: Datasets capturing motion, typing patterns, or device sensor data used in side-channel or behavioral analysis.
        \t\t- 4. **Textual/Data Mining Datasets**: Datasets containing textual or structured data used in tasks such as phishing or scam detection, or scientific literature.
        \t\t- 5. **Tabular / Structured Datasets**: Datasets containing structured numerical or categorical data in tabular form, commonly used for classification or regression tasks (e.g., Iris dataset).

        \t\t\t- Rule: Datasets with explicit cybersecurity semantics should be classified under semantic categories
        (e.g., Attacks, User Activities, User Attributes, etc.), regardless of modality. Modality-based categories apply only when no semantic category fits.
        (e.g., the UCI Adult dataset is tabular but classified as **User Attributes** since 
        it captures user characteristics.)
         Important:
        \t2- The taxonomy below is the primary framework, but the model is not strictly restricted to it.
        \t3- If a dataset does not semantically fit any listed **category** or **subcategory**, you may introduce a new category or subcategory only
        when strongly supported by evidence from the paper.
        \t4- Do NOT force a dataset into an incorrect category.
        \t5- Do NOT assign multiple categories to a single dataset. Each dataset must have exactly one category and subcategory.
        \t6- Do NOT introduce new categories unless the dataset clearly does not fit any existing category in the taxonomy. Only create a new category when it represents a genuinely distinct type of data not covered by existing labels.

        \t7. For each evidence_span, return only one small sentence from the paper.
         Do not return multiple sentences or long passages.
        
        Here are the dataset names extracted in the first task; only these datasets should be considered in the following steps:
        
        {dataset_names}

        ### Output Structure:

        The output must strictly follow this JSON structure:
        
        \t- Example: In the "A Lightweight IoT Cryptojacking Detection Mechanism in Heterogeneous Smart Home Networks", you should return output like:
        "datasets": [
            {{
                "dataset_name": "IoT Cryptojacking Dataset",
                "category": "macro_level_internet_characteristics",
                "subcategory": "network_traces",
                "evidence_span": "we used a dataset of network traces consisting of 6.4M network packets"
            }},
            {{
                dataset_name": "Benign dataset",
                "category": "macro_level_internet_characteristics",
                "subcategory": "network_traces",
                "evidence_span": "we downloaded the benign dataset from a public repository [54]"
            }}
           ]
         }}

      
      <Start of Paper Content>
      {content}
      <End of Paper Content>
      
      Your response: """

    elif task == "dataset_creation_reason":
        return f"""
        You are tasked with identifying the **reason for creating the dataset** in the cybersecurity paper titled "{title}".
        Important:
        - Only answer for datasets whose **action = "created"**.
        - Ignore datasets whose action = "used".
      
        - Focus only on the dataset(s) listed below:
        {dataset_names}

        Rules:
        \t1. Identify why the dataset was created in the paper. Look for statements such as:
        "No publicly available dataset", "Outdated datasets", "Limited coverage", "Need for labeled data",
        "Domain-specific constraints", "Need for large-scale data", "To detect a new type of fraud or attack"
        "Need for synthetic data", "Other reasons menitoned in the paper".
        
        \t2. Select the most applicable reason(s) for dataset creation from the options:
        \t   **(A) No publicly available dataset**  
        \t   **(B) Outdated datasets**  
        \t   **(C) Limited coverage**  
        \t   **(D) Need for labeled data**  
        \t   **(E) Domain-specific constraints**  
        \t   **(F) Need for large-scale data**  
        \t   **(G) To detect a new type of fraud or attack
        \t   **(H) Need for synthetic data**  
        \t   **(I) Novel Dataset**
        \t **(N) None** will be used when "No motivation given" behind the creation of dataset.
        \t   If the dataset creation reason does not fit any predefined category, create a new, specific reason based on the paper (e.g., "To capture real-world IoT traffic", "To address class imbalance in malware detection").
        Avoid generic terms like "other".  


        \t3. If no reason is mentioned or implied at all, return:
        
        {{
            "dataset_creation_reason": "No reason given",
            "evidence_span": ""
        }}

        \t4. Only created datasets will be passed to this prompt. Do not analyze **used** datasets.

        \t5. For each **evidence_span**, return only one small sentence from the paper. 
        Do not return multiple sentences or long passages.
        
        Here is the paper:
        <Start of Paper Content>
        {content}
        <End of Paper Content>
        
        Return valid JSON only in this format:
        {{
          "dataset_creation_reason": [
            {{
              "dataset_name": "",
              "reasons": [
                {{
                "code": "",
                "label": "",
                "evidence_span": ""
                }}
               ]
            }}
          ]
        }}

       
        Your response: """

    else:
        raise ValueError("Invalid task")


In [ ]:
import os
import json
import csv
from openai import OpenAI
from dotenv import load_dotenv

# ================== OPENAI CLIENT ==================

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# ================== HELPERS ==================

def save_incremental_results(results, output_file="results_incremental.jsonl"):
    with open(output_file, "a", encoding="utf-8") as file:
        for paper_id, result in results.items():
            file.write(json.dumps({paper_id: result}, ensure_ascii=False) + "\n")


def load_saved_results(output_file="results_incremental.jsonl"):
    saved_ids = set()
    saved_results = {}

    try:
        with open(output_file, "r", encoding="utf-8") as file:
            for line in file:
                result = json.loads(line.strip())
                for paper_id, data in result.items():
                    saved_ids.add(paper_id)
                    saved_results[paper_id] = data
    except FileNotFoundError:
        print(f"No saved results found in {output_file}. Starting fresh.")

    return saved_ids, saved_results


def load_jsonl(file_path):
    papers = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            papers.append(json.loads(line.strip()))
    return papers


def try_parse_json(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()
    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    try:
        return json.loads(text)
    except Exception:
        return text


# ================== MAIN PROCESSING ==================

def process_papers_for_tasks(papers, start_index=0, output_file="results_incremental.jsonl"):
    task_results = {}

    total_count = len(papers) + start_index

    for i, paper in enumerate(papers, start=start_index):
        paper_id = paper["paper_id"]
        paper_title = paper["title"]

        print(f"\nProcessing paper {i + 1}/{total_count}")
        print(f"Paper ID: {paper_id}")
        print(f"Title: {paper_title}")

        # Skip already processed papers
        if paper_id in processed_ids:
            print(f"Skipping already processed paper: {paper_id} | {paper_title}")
            continue

        task_results[paper_id] = {
            "paper_id": paper_id,
            "title": paper_title
        }

        # ----------------------------------------
        # TASK 1: DATASET NAME
        # ----------------------------------------
        all_dataset_names = []
        created_dataset_names = []

        try:
            user_prompt = generate_system_prompt(paper, "dataset_name", None)
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2,
                max_tokens=5000
            )
            response_text = response.choices[0].message.content
            parsed_response = try_parse_json(response_text)

            print(f"Result for dataset_name: {response_text}")
            task_results[paper_id]["dataset_name"] = parsed_response

            if isinstance(parsed_response, dict):
                datasets = parsed_response.get("datasets", [])

                print("---- Extracted datasets ----")
                for d in datasets:
                    name = d.get("dataset_name")
                    action = d.get("action", d.get("Action", ""))

                    if isinstance(action, str):
                        action = action.strip().lower()

                    print("Dataset:", name, "| Action:", action)

                    if name:
                        all_dataset_names.append(name)

                    if name and action == "created":
                        created_dataset_names.append(name)

                print("All dataset names:", all_dataset_names)
                print("Created dataset names:", created_dataset_names)

        except Exception as e:
            print(f"Error processing dataset_name for paper {i + 1}: {e}")
            task_results[paper_id]["dataset_name"] = f"error: {str(e)}"

        # ----------------------------------------
        # TASK 2: DATASET CATEGORIES
        # ----------------------------------------
        if all_dataset_names:
            try:
                print("Passing to dataset_categories:", all_dataset_names)

                user_prompt = generate_system_prompt(
                    paper,
                    "dataset_categories",
                    all_dataset_names
                )
                response = client.chat.completions.create(
                    model="gpt-5-mini",
                    messages=[
                        {"role": "user", "content": user_prompt}
                    ],
                    max_completion_tokens=5000
                )
                response_text = response.choices[0].message.content
                parsed_response = try_parse_json(response_text)

                print(f"Result for dataset_categories: {response_text}")
                task_results[paper_id]["dataset_categories"] = parsed_response

            except Exception as e:
                print(f"Error processing dataset_categories for paper {i + 1}: {e}")
                task_results[paper_id]["dataset_categories"] = f"error: {str(e)}"

        # ----------------------------------------
        # TASK 3: DATASET CREATION REASON
        # ----------------------------------------
        if created_dataset_names:
            try:
                print("Passing to dataset_creation_reason:", created_dataset_names)

                user_prompt = generate_system_prompt(
                    paper,
                    "dataset_creation_reason",
                    created_dataset_names
                )
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.2,
                    max_tokens=5000
                )
                response_text = response.choices[0].message.content
                parsed_response = try_parse_json(response_text)

                print(f"Result for dataset_creation_reason: {response_text}")
                task_results[paper_id]["dataset_creation_reason"] = parsed_response

            except Exception as e:
                print(f"Error processing dataset_creation_reason for paper {i + 1}: {e}")
                task_results[paper_id]["dataset_creation_reason"] = f"error: {str(e)}"

        # ----------------------------------------
        # TASK 4: DATA SOURCES
        # ----------------------------------------
        if created_dataset_names:
            try:
                print("Passing to data_sources:", created_dataset_names)

                user_prompt = generate_system_prompt(
                    paper,
                    "data_sources",
                    created_dataset_names
                )
                response = client.chat.completions.create(
                    model="gpt-5-mini",
                    messages=[
                        {"role": "user", "content": user_prompt}
                    ],
                    max_completion_tokens=5000
                )
                response_text = response.choices[0].message.content
                parsed_response = try_parse_json(response_text)

                print(f"Result for data_sources: {response_text}")
                task_results[paper_id]["data_sources"] = parsed_response

            except Exception as e:
                print(f"Error processing data_sources for paper {i + 1}: {e}")
                task_results[paper_id]["data_sources"] = f"error: {str(e)}"

        # Save incremental results
        save_incremental_results({paper_id: task_results[paper_id]}, output_file)

    return task_results


# ================== LOAD SAVED RESULTS ==================

output_file = "results.jsonl"
processed_ids, processed_results = load_saved_results(output_file)

# ================== LOAD PAPERS ==================

input_file = "new_pipeline_all_dataset_papers.jsonl"
all_papers = load_jsonl(input_file)

remaining_papers = [
    paper for paper in all_papers
    if paper["paper_id"] not in processed_ids
]

# ================== RUN ==================

all_results = process_papers_for_tasks(
    remaining_papers,
    start_index=len(processed_ids),
    output_file=output_file
)

processed_results.update(all_results)

# ================== SAVE FINAL RESULTS ==================

output_csv = "results.csv"
output_jsonl = "results.jsonl"

task_columns = [
    "dataset_name",
    "dataset_categories",
    "dataset_creation_reason"
]

# Save CSV
with open(output_csv, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["paper_id", "title"] + task_columns)

    for paper_id, results in processed_results.items():
        row = [
            results.get("paper_id", paper_id),
            results.get("title", "")
        ]

        for task in task_columns:
            task_result = results.get(task, "No result")
            if isinstance(task_result, (dict, list)):
                task_result = json.dumps(task_result, ensure_ascii=False)
            row.append(task_result)

        writer.writerow(row)

# Save JSONL
with open(output_jsonl, "w", encoding="utf-8") as jsonl_file:
    for paper_id, results in processed_results.items():
        jsonl_file.write(json.dumps({paper_id: results}, ensure_ascii=False) + "\n")

print(f"\nFinal results saved to {output_csv} and {output_jsonl}.")

Domain Analysis

In [6]:

import os
import json
from openai import OpenAI
from dotenv import load_dotenv



load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

acm_ccs_structure = [
  {
    "path": "General and reference > Document types > Surveys and overviews",
    "high_level_domain": "General and reference",
    "subdomain": "Document types",
    "node_3": "Surveys and overviews",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Document types > Reference works",
    "high_level_domain": "General and reference",
    "subdomain": "Document types",
    "node_3": "Reference works",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Document types > General conference proceedings",
    "high_level_domain": "General and reference",
    "subdomain": "Document types",
    "node_3": "General conference proceedings",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Document types > Biographies",
    "high_level_domain": "General and reference",
    "subdomain": "Document types",
    "node_3": "Biographies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Document types > General literature",
    "high_level_domain": "General and reference",
    "subdomain": "Document types",
    "node_3": "General literature",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Document types > Computing standards, RFCs and guidelines",
    "high_level_domain": "General and reference",
    "subdomain": "Document types",
    "node_3": "Computing standards, RFCs and guidelines",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Reliability",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Reliability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Empirical studies",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Empirical studies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Measurement",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Measurement",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Metrics",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Metrics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Evaluation",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Evaluation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Experimentation",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Experimentation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Estimation",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Estimation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Design",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Design",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Performance",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Performance",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Validation",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Validation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "General and reference > Cross-computing tools and techniques > Verification",
    "high_level_domain": "General and reference",
    "subdomain": "Cross-computing tools and techniques",
    "node_3": "Verification",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Printed circuit boards > Electromagnetic interference and compatibility",
    "high_level_domain": "Hardware",
    "subdomain": "Printed circuit boards",
    "node_3": "Electromagnetic interference and compatibility",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Printed circuit boards > PCB design and layout",
    "high_level_domain": "Hardware",
    "subdomain": "Printed circuit boards",
    "node_3": "PCB design and layout",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Signal processing systems",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Signal processing systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Signal processing systems > Digital signal processing",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Signal processing systems",
    "node_4": "Digital signal processing",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Signal processing systems > Beamforming",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Signal processing systems",
    "node_4": "Beamforming",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Signal processing systems > Noise reduction",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Signal processing systems",
    "node_4": "Noise reduction",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Sensors and actuators",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Sensors and actuators",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Buses and high-speed links",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Buses and high-speed links",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Displays and imagers",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Displays and imagers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > External storage",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "External storage",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Networking hardware",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Networking hardware",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Printers",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Printers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Sensor applications and deployments",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Sensor applications and deployments",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Sensor devices and platforms",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Sensor devices and platforms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Sound-based input / output",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Sound-based input / output",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Tactile and hand-based interfaces",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Tactile and hand-based interfaces",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Tactile and hand-based interfaces > Touch screens",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Tactile and hand-based interfaces",
    "node_4": "Touch screens",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Tactile and hand-based interfaces > Haptic devices",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Tactile and hand-based interfaces",
    "node_4": "Haptic devices",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Scanners",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Scanners",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Wireless devices",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Wireless devices",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Wireless integrated network sensors",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Wireless integrated network sensors",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Communication hardware, interfaces and storage > Electro-mechanical devices",
    "high_level_domain": "Hardware",
    "subdomain": "Communication hardware, interfaces and storage",
    "node_3": "Electro-mechanical devices",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > 3D integrated circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "3D integrated circuits",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Interconnect",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Interconnect",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Interconnect > Input / output circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Interconnect",
    "node_4": "Input / output circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Interconnect > Metallic interconnect",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Interconnect",
    "node_4": "Metallic interconnect",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Interconnect > Photonic and optical interconnect",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Interconnect",
    "node_4": "Photonic and optical interconnect",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Interconnect > Radio frequency and wireless interconnect",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Interconnect",
    "node_4": "Radio frequency and wireless interconnect",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Semiconductor memory",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Semiconductor memory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Semiconductor memory > Dynamic memory",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Semiconductor memory",
    "node_4": "Dynamic memory",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Semiconductor memory > Static memory",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Semiconductor memory",
    "node_4": "Static memory",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Semiconductor memory > Non-volatile memory",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Semiconductor memory",
    "node_4": "Non-volatile memory",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Semiconductor memory > Read-only memory",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Semiconductor memory",
    "node_4": "Read-only memory",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Digital switches",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Digital switches",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Digital switches > Transistors",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Digital switches",
    "node_4": "Transistors",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Digital switches > Logic families",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Digital switches",
    "node_4": "Logic families",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Logic circuit",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Logic circuit",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Logic circuit > Arithmetic and datapath circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Logic circuit",
    "node_4": "Arithmetic and datapath circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Logic circuit > Asynchronous circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Logic circuit",
    "node_4": "Asynchronous circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Logic circuit > Combinational circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Logic circuit",
    "node_4": "Combinational circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Logic circuit > Design modules and hierarchy",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Logic circuit",
    "node_4": "Design modules and hierarchy",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Logic circuit > Finite state machines",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Logic circuit",
    "node_4": "Finite state machines",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Logic circuit > Sequential circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Logic circuit",
    "node_4": "Sequential circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Reconfigurable logic and FPGAs",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Reconfigurable logic and FPGAs",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Reconfigurable logic and FPGAs > Hardware accelerators",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Reconfigurable logic and FPGAs",
    "node_4": "Hardware accelerators",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Reconfigurable logic and FPGAs > High-speed input / output",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Reconfigurable logic and FPGAs",
    "node_4": "High-speed input / output",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Reconfigurable logic and FPGAs > Programmable logic elements",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Reconfigurable logic and FPGAs",
    "node_4": "Programmable logic elements",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Reconfigurable logic and FPGAs > Programmable interconnect",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Reconfigurable logic and FPGAs",
    "node_4": "Programmable interconnect",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Reconfigurable logic and FPGAs > Reconfigurable logic applications",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Reconfigurable logic and FPGAs",
    "node_4": "Reconfigurable logic applications",
    "node_5": ""
  },
  {
    "path": "Hardware > Integrated circuits > Reconfigurable logic and FPGAs > Evolvable hardware",
    "high_level_domain": "Hardware",
    "subdomain": "Integrated circuits",
    "node_3": "Reconfigurable logic and FPGAs",
    "node_4": "Evolvable hardware",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > 3D integrated circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "3D integrated circuits",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "Data conversion",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "Clock generation and timing",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "Analog and mixed-signal circuit optimization",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "Radio frequency and wireless circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "Wireline communication",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "Analog and mixed-signal circuit synthesis",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Application-specific VLSI designs",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Application-specific VLSI designs > Application specific integrated circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Application-specific VLSI designs",
    "node_4": "Application specific integrated circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Application-specific VLSI designs > Application specific instruction set processors",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Application-specific VLSI designs",
    "node_4": "Application specific instruction set processors",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Application-specific VLSI designs > Application specific processors",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Application-specific VLSI designs",
    "node_4": "Application specific processors",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Analog and mixed-signal circuits > Analog and mixed-signal circuit synthesis",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Analog and mixed-signal circuits",
    "node_4": "Analog and mixed-signal circuit synthesis",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Design reuse and communication-based design",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Design reuse and communication-based design",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Design reuse and communication-based design > Network on chip",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Design reuse and communication-based design",
    "node_4": "Network on chip",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Design reuse and communication-based design > System on a chip",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Design reuse and communication-based design",
    "node_4": "System on a chip",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Design reuse and communication-based design > Platform-based design",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Design reuse and communication-based design",
    "node_4": "Platform-based design",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Design reuse and communication-based design > Hard and soft IP",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Design reuse and communication-based design",
    "node_4": "Hard and soft IP",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Design rules",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Design rules",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Economics of chip design and manufacturing",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Economics of chip design and manufacturing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Full-custom circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Full-custom circuits",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > VLSI design manufacturing considerations",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "VLSI design manufacturing considerations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > On-chip resource management",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "On-chip resource management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > Standard cell libraries",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "Standard cell libraries",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > VLSI packaging",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "VLSI packaging",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > VLSI packaging > Die and wafer stacking",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "VLSI packaging",
    "node_4": "Die and wafer stacking",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > VLSI packaging > Input / output styles",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "VLSI packaging",
    "node_4": "Input / output styles",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > VLSI packaging > Multi-chip modules",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "VLSI packaging",
    "node_4": "Multi-chip modules",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > VLSI packaging > Package-level interconnect",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "VLSI packaging",
    "node_4": "Package-level interconnect",
    "node_5": ""
  },
  {
    "path": "Hardware > Very large scale integration design > VLSI system specification and constraints",
    "high_level_domain": "Hardware",
    "subdomain": "Very large scale integration design",
    "node_3": "VLSI system specification and constraints",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Thermal issues",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Thermal issues",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Thermal issues > Temperature monitoring",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Thermal issues",
    "node_4": "Temperature monitoring",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Thermal issues > Temperature simulation",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Thermal issues",
    "node_4": "Temperature simulation",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Thermal issues > Temperature control",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Thermal issues",
    "node_4": "Temperature control",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Thermal issues > Temperature optimization",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Thermal issues",
    "node_4": "Temperature optimization",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy generation and storage",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy generation and storage",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy generation and storage > Batteries",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy generation and storage",
    "node_4": "Batteries",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy generation and storage > Fuel-based energy",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy generation and storage",
    "node_4": "Fuel-based energy",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy generation and storage > Renewable energy",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy generation and storage",
    "node_4": "Renewable energy",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy generation and storage > Reusable energy storage",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy generation and storage",
    "node_4": "Reusable energy storage",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy distribution",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy distribution",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy distribution > Energy metering",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy distribution",
    "node_4": "Energy metering",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy distribution > Power conversion",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy distribution",
    "node_4": "Power conversion",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy distribution > Power networks",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy distribution",
    "node_4": "Power networks",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Energy distribution > Smart grid",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Energy distribution",
    "node_4": "Smart grid",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Impact on the environment",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Impact on the environment",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Power estimation and optimization",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Power estimation and optimization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Power estimation and optimization > Switching devices power issues",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Power estimation and optimization",
    "node_4": "Switching devices power issues",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Power estimation and optimization > Interconnect power issues",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Power estimation and optimization",
    "node_4": "Interconnect power issues",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Power estimation and optimization > Circuits power issues",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Power estimation and optimization",
    "node_4": "Circuits power issues",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Power estimation and optimization > Chip-level power issues",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Power estimation and optimization",
    "node_4": "Chip-level power issues",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Power estimation and optimization > Platform power issues",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Power estimation and optimization",
    "node_4": "Platform power issues",
    "node_5": ""
  },
  {
    "path": "Hardware > Power and energy > Power estimation and optimization > Enterprise level and data centers power issues",
    "high_level_domain": "Hardware",
    "subdomain": "Power and energy",
    "node_3": "Power estimation and optimization",
    "node_4": "Enterprise level and data centers power issues",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > High-level and register-transfer level synthesis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "High-level and register-transfer level synthesis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > High-level and register-transfer level synthesis > Datapath optimization",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "High-level and register-transfer level synthesis",
    "node_4": "Datapath optimization",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > High-level and register-transfer level synthesis > Hardware-software codesign",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "High-level and register-transfer level synthesis",
    "node_4": "Hardware-software codesign",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > High-level and register-transfer level synthesis > Resource binding and sharing",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "High-level and register-transfer level synthesis",
    "node_4": "Resource binding and sharing",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > High-level and register-transfer level synthesis > Operations scheduling",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "High-level and register-transfer level synthesis",
    "node_4": "Operations scheduling",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Hardware description languages and compilation",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Hardware description languages and compilation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Hardware description languages and compilation > Logic synthesis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Hardware description languages and compilation",
    "node_4": "Logic synthesis",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Hardware description languages and compilation > Combinational synthesis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Hardware description languages and compilation",
    "node_4": "Combinational synthesis",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Hardware description languages and compilation > Circuit optimization",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Hardware description languages and compilation",
    "node_4": "Circuit optimization",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Hardware description languages and compilation > Sequential synthesis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Hardware description languages and compilation",
    "node_4": "Sequential synthesis",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Hardware description languages and compilation > Technology-mapping",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Hardware description languages and compilation",
    "node_4": "Technology-mapping",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Hardware description languages and compilation > Transistor-level synthesis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Hardware description languages and compilation",
    "node_4": "Transistor-level synthesis",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Timing analysis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Timing analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Timing analysis > Electrical-level simulation",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Timing analysis",
    "node_4": "Electrical-level simulation",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Timing analysis > Model-order reduction",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Timing analysis",
    "node_4": "Model-order reduction",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Timing analysis > Compact delay models",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Timing analysis",
    "node_4": "Compact delay models",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Timing analysis > Static timing analysis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Timing analysis",
    "node_4": "Static timing analysis",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Timing analysis > Statistical timing analysis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Timing analysis",
    "node_4": "Statistical timing analysis",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Timing analysis > Transition-based timing analysis",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Timing analysis",
    "node_4": "Transition-based timing analysis",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Methodologies for EDA",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Methodologies for EDA",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Methodologies for EDA > Best practices for EDA",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Methodologies for EDA",
    "node_4": "Best practices for EDA",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Methodologies for EDA > Design databases for EDA",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Methodologies for EDA",
    "node_4": "Design databases for EDA",
    "node_5": ""
  },
  {
    "path": "Hardware > Electronic design automation > Methodologies for EDA > Software tools for EDA",
    "high_level_domain": "Hardware",
    "subdomain": "Electronic design automation",
    "node_3": "Methodologies for EDA",
    "node_4": "Software tools for EDA",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Model checking",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Model checking",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Coverage metrics",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Coverage metrics",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Equivalence checking",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Equivalence checking",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Semi-formal verification",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Semi-formal verification",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Simulation and emulation",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Simulation and emulation",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Transaction-level verification",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Transaction-level verification",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Theorem proving and SAT solving",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Theorem proving and SAT solving",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Functional verification > Assertion checking",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Functional verification",
    "node_4": "Assertion checking",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Physical verification",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Physical verification",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Physical verification > Design rule checking",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Physical verification",
    "node_4": "Design rule checking",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Physical verification > Layout-versus-schematics",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Physical verification",
    "node_4": "Layout-versus-schematics",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Physical verification > Power and thermal analysis",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Physical verification",
    "node_4": "Power and thermal analysis",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Physical verification > Timing analysis and sign-off",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Physical verification",
    "node_4": "Timing analysis and sign-off",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Post-manufacture validation and debug",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Post-manufacture validation and debug",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Post-manufacture validation and debug > Bug detection, localization and diagnosis",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Post-manufacture validation and debug",
    "node_4": "Bug detection, localization and diagnosis",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Post-manufacture validation and debug > Bug fixing (hardware)",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Post-manufacture validation and debug",
    "node_4": "Bug fixing (hardware)",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware validation > Post-manufacture validation and debug > Design for debug",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware validation",
    "node_3": "Post-manufacture validation and debug",
    "node_4": "Design for debug",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Analog, mixed-signal and radio frequency test",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Analog, mixed-signal and radio frequency test",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Board- and system-level test",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Board- and system-level test",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Defect-based test",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Defect-based test",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Design for testability",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Design for testability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Design for testability > Built-in self-test",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Design for testability",
    "node_4": "Built-in self-test",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Design for testability > Online test and diagnostics",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Design for testability",
    "node_4": "Online test and diagnostics",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Design for testability > Test data compression",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Design for testability",
    "node_4": "Test data compression",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Fault models and test metrics",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Fault models and test metrics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Memory test and repair",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Memory test and repair",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Hardware reliability screening",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Hardware reliability screening",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Test-pattern generation and fault simulation",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Test-pattern generation and fault simulation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Hardware test > Testing with distributed and parallel systems",
    "high_level_domain": "Hardware",
    "subdomain": "Hardware test",
    "node_3": "Testing with distributed and parallel systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Fault tolerance",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Fault tolerance",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Fault tolerance > Error detection and error correction",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Fault tolerance",
    "node_4": "Error detection and error correction",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Fault tolerance > Failure prediction",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Fault tolerance",
    "node_4": "Failure prediction",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Fault tolerance > Failure recovery, maintenance and self-repair",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Fault tolerance",
    "node_4": "Failure recovery, maintenance and self-repair",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Fault tolerance > Redundancy",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Fault tolerance",
    "node_4": "Redundancy",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Fault tolerance > Self-checking mechanisms",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Fault tolerance",
    "node_4": "Self-checking mechanisms",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Fault tolerance > System-level fault tolerance",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Fault tolerance",
    "node_4": "System-level fault tolerance",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Design for manufacturability",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Design for manufacturability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Design for manufacturability > Process variations",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Design for manufacturability",
    "node_4": "Process variations",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Design for manufacturability > Yield and cost modeling",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Design for manufacturability",
    "node_4": "Yield and cost modeling",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Design for manufacturability > Yield and cost optimization",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Design for manufacturability",
    "node_4": "Yield and cost optimization",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Hardware reliability",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Hardware reliability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Hardware reliability > Aging of circuits and systems",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Hardware reliability",
    "node_4": "Aging of circuits and systems",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Hardware reliability > Circuit hardening",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Hardware reliability",
    "node_4": "Circuit hardening",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Hardware reliability > Early-life failures and infant mortality",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Hardware reliability",
    "node_4": "Early-life failures and infant mortality",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Hardware reliability > Process, voltage and temperature variations",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Hardware reliability",
    "node_4": "Process, voltage and temperature variations",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Hardware reliability > Signal integrity and noise analysis",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Hardware reliability",
    "node_4": "Signal integrity and noise analysis",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Hardware reliability > Transient errors and upsets",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Hardware reliability",
    "node_4": "Transient errors and upsets",
    "node_5": ""
  },
  {
    "path": "Hardware > Robustness > Safety critical systems",
    "high_level_domain": "Hardware",
    "subdomain": "Robustness",
    "node_3": "Safety critical systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Analysis and design of emerging devices and systems",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Analysis and design of emerging devices and systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Analysis and design of emerging devices and systems > Emerging architectures",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Analysis and design of emerging devices and systems",
    "node_4": "Emerging architectures",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Analysis and design of emerging devices and systems > Emerging languages and compilers",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Analysis and design of emerging devices and systems",
    "node_4": "Emerging languages and compilers",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Analysis and design of emerging devices and systems > Emerging simulation",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Analysis and design of emerging devices and systems",
    "node_4": "Emerging simulation",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Analysis and design of emerging devices and systems > Emerging tools and methodologies",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Analysis and design of emerging devices and systems",
    "node_4": "Emerging tools and methodologies",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Biology-related information processing",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Biology-related information processing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Biology-related information processing > Bio-embedded electronics",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Biology-related information processing",
    "node_4": "Bio-embedded electronics",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Biology-related information processing > Neural systems",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Biology-related information processing",
    "node_4": "Neural systems",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Circuit substrates",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Circuit substrates",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Circuit substrates > III-V compounds",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Circuit substrates",
    "node_4": "III-V compounds",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Circuit substrates > Carbon based electronics",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Circuit substrates",
    "node_4": "Carbon based electronics",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Circuit substrates > Cellular neural networks",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Circuit substrates",
    "node_4": "Cellular neural networks",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Circuit substrates > Flexible and printable circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Circuit substrates",
    "node_4": "Flexible and printable circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Circuit substrates > Superconducting circuits",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Circuit substrates",
    "node_4": "Superconducting circuits",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Electromechanical systems",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Electromechanical systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Electromechanical systems > Microelectromechanical systems",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Electromechanical systems",
    "node_4": "Microelectromechanical systems",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Electromechanical systems > Nanoelectromechanical systems",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Electromechanical systems",
    "node_4": "Nanoelectromechanical systems",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Emerging interfaces",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Emerging interfaces",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Memory and dense storage",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Memory and dense storage",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Emerging optical and photonic technologies",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Emerging optical and photonic technologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Reversible logic",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Reversible logic",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Plasmonics",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Plasmonics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Quantum technologies",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Quantum technologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Quantum technologies > Single electron devices",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Quantum technologies",
    "node_4": "Single electron devices",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Quantum technologies > Tunneling devices",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Quantum technologies",
    "node_4": "Tunneling devices",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Quantum technologies > Quantum computation",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Quantum technologies",
    "node_4": "Quantum computation",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Quantum technologies > Quantum communication and cryptography",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Quantum technologies",
    "node_4": "Quantum communication and cryptography",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Quantum technologies > Quantum error correction and fault tolerance",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Quantum technologies",
    "node_4": "Quantum error correction and fault tolerance",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Quantum technologies > Quantum dots and cellular automata",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Quantum technologies",
    "node_4": "Quantum dots and cellular automata",
    "node_5": ""
  },
  {
    "path": "Hardware > Emerging technologies > Spintronics and magnetic technologies",
    "high_level_domain": "Hardware",
    "subdomain": "Emerging technologies",
    "node_3": "Spintronics and magnetic technologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Serial architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Serial architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Serial architectures > Reduced instruction set computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Serial architectures",
    "node_4": "Reduced instruction set computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Serial architectures > Complex instruction set computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Serial architectures",
    "node_4": "Complex instruction set computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Serial architectures > Superscalar architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Serial architectures",
    "node_4": "Superscalar architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Serial architectures > Pipeline computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Serial architectures",
    "node_4": "Pipeline computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Serial architectures > Stack machines",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Serial architectures",
    "node_4": "Stack machines",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Very long instruction word",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Very long instruction word",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Interconnection architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Interconnection architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Multiple instruction, multiple data",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Multiple instruction, multiple data",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Cellular architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Cellular architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Multiple instruction, single data",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Multiple instruction, single data",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Single instruction, multiple data",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Single instruction, multiple data",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Systolic arrays",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Systolic arrays",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Parallel architectures > Multicore architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Parallel architectures",
    "node_4": "Multicore architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Distributed architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Distributed architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Distributed architectures > Cloud computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Distributed architectures",
    "node_4": "Cloud computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Distributed architectures > Client-server architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Distributed architectures",
    "node_4": "Client-server architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Distributed architectures > n-tier architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Distributed architectures",
    "node_4": "n-tier architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Distributed architectures > Peer-to-peer architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Distributed architectures",
    "node_4": "Peer-to-peer architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Distributed architectures > Grid computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Distributed architectures",
    "node_4": "Grid computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Neural networks",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Neural networks",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Reconfigurable computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Reconfigurable computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Analog computers",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Analog computers",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Data flow architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Data flow architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Heterogeneous (hybrid) systems",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Heterogeneous (hybrid) systems",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Self-organizing autonomic computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Self-organizing autonomic computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Optical computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Optical computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Quantum computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Quantum computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Molecular computing",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Molecular computing",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > High-level language architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "High-level language architectures",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Architectures > Other architectures > Special purpose systems",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Architectures",
    "node_3": "Other architectures",
    "node_4": "Special purpose systems",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Sensor networks",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Sensor networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Robotics",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Robotics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Robotics > Robotic components",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Robotics",
    "node_4": "Robotic components",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Robotics > Robotic control",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Robotics",
    "node_4": "Robotic control",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Robotics > Evolutionary robotics",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Robotics",
    "node_4": "Evolutionary robotics",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Robotics > Robotic autonomy",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Robotics",
    "node_4": "Robotic autonomy",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Robotics > External interfaces for robotics",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Robotics",
    "node_4": "External interfaces for robotics",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Sensors and actuators",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Sensors and actuators",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > System on a chip",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "System on a chip",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Embedded systems",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Embedded systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Embedded systems > Firmware",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Embedded systems",
    "node_4": "Firmware",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Embedded systems > Embedded hardware",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Embedded systems",
    "node_4": "Embedded hardware",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Embedded and cyber-physical systems > Embedded systems > Embedded software",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Embedded and cyber-physical systems",
    "node_3": "Embedded systems",
    "node_4": "Embedded software",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Real-time systems > Real-time operating systems",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Real-time systems",
    "node_3": "Real-time operating systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Real-time systems > Real-time languages",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Real-time systems",
    "node_3": "Real-time languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Real-time systems > Real-time system specification",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Real-time systems",
    "node_3": "Real-time system specification",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Real-time systems > Real-time system architecture",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Real-time systems",
    "node_3": "Real-time system architecture",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Dependable and fault-tolerant systems and networks > Reliability",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Dependable and fault-tolerant systems and networks",
    "node_3": "Reliability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Dependable and fault-tolerant systems and networks > Availability",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Dependable and fault-tolerant systems and networks",
    "node_3": "Availability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Dependable and fault-tolerant systems and networks > Maintainability and maintenance",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Dependable and fault-tolerant systems and networks",
    "node_3": "Maintainability and maintenance",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Dependable and fault-tolerant systems and networks > Processors and memory architectures",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Dependable and fault-tolerant systems and networks",
    "node_3": "Processors and memory architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Dependable and fault-tolerant systems and networks > Secondary storage organization",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Dependable and fault-tolerant systems and networks",
    "node_3": "Secondary storage organization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Dependable and fault-tolerant systems and networks > Redundancy",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Dependable and fault-tolerant systems and networks",
    "node_3": "Redundancy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computer systems organization > Dependable and fault-tolerant systems and networks > Fault-tolerant network topologies",
    "high_level_domain": "Computer systems organization",
    "subdomain": "Dependable and fault-tolerant systems and networks",
    "node_3": "Fault-tolerant network topologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network architectures > Network design principles",
    "high_level_domain": "Networks",
    "subdomain": "Network architectures",
    "node_3": "Network design principles",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network architectures > Network design principles > Layering",
    "high_level_domain": "Networks",
    "subdomain": "Network architectures",
    "node_3": "Network design principles",
    "node_4": "Layering",
    "node_5": ""
  },
  {
    "path": "Networks > Network architectures > Network design principles > Naming and addressing",
    "high_level_domain": "Networks",
    "subdomain": "Network architectures",
    "node_3": "Network design principles",
    "node_4": "Naming and addressing",
    "node_5": ""
  },
  {
    "path": "Networks > Network architectures > Programming interfaces",
    "high_level_domain": "Networks",
    "subdomain": "Network architectures",
    "node_3": "Programming interfaces",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Network protocol design",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Network protocol design",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Protocol correctness",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Protocol correctness",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Protocol correctness > Protocol testing and verification",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Protocol correctness",
    "node_4": "Protocol testing and verification",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Protocol correctness > Formal specifications",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Protocol correctness",
    "node_4": "Formal specifications",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Link-layer protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Link-layer protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Network layer protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Network layer protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Network layer protocols > Routing protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Network layer protocols",
    "node_4": "Routing protocols",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Network layer protocols > Signaling protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Network layer protocols",
    "node_4": "Signaling protocols",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Transport protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Transport protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Session protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Session protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Presentation protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Presentation protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Application layer protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Application layer protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Application layer protocols > Peer-to-peer protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Application layer protocols",
    "node_4": "Peer-to-peer protocols",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > OAM protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "OAM protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > OAM protocols > Time synchronization protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "OAM protocols",
    "node_4": "Time synchronization protocols",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > OAM protocols > Network policy",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "OAM protocols",
    "node_4": "Network policy",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Cross-layer protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Cross-layer protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network protocols > Network File System (NFS) protocol",
    "high_level_domain": "Networks",
    "subdomain": "Network protocols",
    "node_3": "Network File System (NFS) protocol",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Intermediate nodes",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Intermediate nodes",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Intermediate nodes > Routers",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Intermediate nodes",
    "node_4": "Routers",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Intermediate nodes > Bridges and switches",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Intermediate nodes",
    "node_4": "Bridges and switches",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Physical links",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Physical links",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Physical links > Repeaters",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Physical links",
    "node_4": "Repeaters",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Middle boxes / network appliances",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Middle boxes / network appliances",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > End nodes",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "End nodes",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > End nodes > Network adapters",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "End nodes",
    "node_4": "Network adapters",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > End nodes > Network servers",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "End nodes",
    "node_4": "Network servers",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Wireless access points, base stations and infrastructure",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Wireless access points, base stations and infrastructure",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Wireless access points, base stations and infrastructure > Cognitive radios",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Wireless access points, base stations and infrastructure",
    "node_4": "Cognitive radios",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Logical nodes",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Logical nodes",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network components > Logical nodes > Network domains",
    "high_level_domain": "Networks",
    "subdomain": "Network components",
    "node_3": "Logical nodes",
    "node_4": "Network domains",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Data path algorithms",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Data path algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Data path algorithms > Packet classification",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Data path algorithms",
    "node_4": "Packet classification",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Data path algorithms > Deep packet inspection",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Data path algorithms",
    "node_4": "Deep packet inspection",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Data path algorithms > Packet scheduling",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Data path algorithms",
    "node_4": "Packet scheduling",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Control path algorithms",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Control path algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Control path algorithms > Network resources allocation",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Control path algorithms",
    "node_4": "Network resources allocation",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Control path algorithms > Network control algorithms",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Control path algorithms",
    "node_4": "Network control algorithms",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Control path algorithms > Traffic engineering algorithms",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Control path algorithms",
    "node_4": "Traffic engineering algorithms",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Control path algorithms > Network design and planning algorithms",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Control path algorithms",
    "node_4": "Network design and planning algorithms",
    "node_5": ""
  },
  {
    "path": "Networks > Network algorithms > Network economics",
    "high_level_domain": "Networks",
    "subdomain": "Network algorithms",
    "node_3": "Network economics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network performance evaluation > Network performance modeling",
    "high_level_domain": "Networks",
    "subdomain": "Network performance evaluation",
    "node_3": "Network performance modeling",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network performance evaluation > Network simulations",
    "high_level_domain": "Networks",
    "subdomain": "Network performance evaluation",
    "node_3": "Network simulations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network performance evaluation > Network experimentation",
    "high_level_domain": "Networks",
    "subdomain": "Network performance evaluation",
    "node_3": "Network experimentation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network performance evaluation > Network performance analysis",
    "high_level_domain": "Networks",
    "subdomain": "Network performance evaluation",
    "node_3": "Network performance analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network performance evaluation > Network measurement",
    "high_level_domain": "Networks",
    "subdomain": "Network performance evaluation",
    "node_3": "Network measurement",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network security",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network security > Security protocols",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network security",
    "node_4": "Security protocols",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network security > Web protocol security",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network security",
    "node_4": "Web protocol security",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network security > Mobile and wireless security",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network security",
    "node_4": "Mobile and wireless security",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network security > Denial-of-service attacks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network security",
    "node_4": "Denial-of-service attacks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network security > Firewalls",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network security",
    "node_4": "Firewalls",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network range",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network range",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network range > Short-range networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network range",
    "node_4": "Short-range networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network range > Local area networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network range",
    "node_4": "Local area networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network range > Metropolitan area networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network range",
    "node_4": "Metropolitan area networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network range > Wide area networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network range",
    "node_4": "Wide area networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network range > Very long-range networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network range",
    "node_4": "Very long-range networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Topology analysis and generation",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Topology analysis and generation",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Physical topologies",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Physical topologies",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Logical / virtual topologies",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Logical / virtual topologies",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Network topology types",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Network topology types",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Point-to-point networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Point-to-point networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Bus networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Bus networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Star networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Star networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Ring networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Ring networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Token ring networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Token ring networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Fiber distributed data interface (FDDI)",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Fiber distributed data interface (FDDI)",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Mesh networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Mesh networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Wireless mesh networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Wireless mesh networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network structure > Hybrid networks",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network structure",
    "node_4": "Hybrid networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network dynamics",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network dynamics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network reliability",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network reliability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network reliability > Error detection and error correction",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network reliability",
    "node_4": "Error detection and error correction",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network mobility",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network mobility",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network manageability",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network manageability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network properties > Network privacy and anonymity",
    "high_level_domain": "Networks",
    "subdomain": "Network properties",
    "node_3": "Network privacy and anonymity",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network services > Naming and addressing",
    "high_level_domain": "Networks",
    "subdomain": "Network services",
    "node_3": "Naming and addressing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network services > Cloud computing",
    "high_level_domain": "Networks",
    "subdomain": "Network services",
    "node_3": "Cloud computing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network services > Location based services",
    "high_level_domain": "Networks",
    "subdomain": "Network services",
    "node_3": "Location based services",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network services > Programmable networks",
    "high_level_domain": "Networks",
    "subdomain": "Network services",
    "node_3": "Programmable networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network services > In-network processing",
    "high_level_domain": "Networks",
    "subdomain": "Network services",
    "node_3": "In-network processing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network services > Network management",
    "high_level_domain": "Networks",
    "subdomain": "Network services",
    "node_3": "Network management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network services > Network monitoring",
    "high_level_domain": "Networks",
    "subdomain": "Network services",
    "node_3": "Network monitoring",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Network on chip",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Network on chip",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Home networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Home networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Storage area networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Storage area networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Data center networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Data center networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Wired access networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Wired access networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Cyber-physical networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Cyber-physical networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Cyber-physical networks > Sensor networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Cyber-physical networks",
    "node_4": "Sensor networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Mobile networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Mobile networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Overlay and other logical network structures",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Overlay and other logical network structures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Overlay and other logical network structures > Peer-to-peer networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Overlay and other logical network structures",
    "node_4": "Peer-to-peer networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Overlay and other logical network structures > World Wide Web (network structure)",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Overlay and other logical network structures",
    "node_4": "World Wide Web (network structure)",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Overlay and other logical network structures > Social media networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Overlay and other logical network structures",
    "node_4": "Social media networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Overlay and other logical network structures > Online social networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Overlay and other logical network structures",
    "node_4": "Online social networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Wireless access networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Wireless access networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Wireless access networks > Wireless local area networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Wireless access networks",
    "node_4": "Wireless local area networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Wireless access networks > Wireless personal area networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Wireless access networks",
    "node_4": "Wireless personal area networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Ad hoc networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Ad hoc networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Ad hoc networks > Mobile ad hoc networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Ad hoc networks",
    "node_4": "Mobile ad hoc networks",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Public Internet",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Public Internet",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Networks > Network types > Packet-switching networks",
    "high_level_domain": "Networks",
    "subdomain": "Network types",
    "node_3": "Packet-switching networks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Contextual software domains",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Contextual software domains",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > E-commerce infrastructure",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "E-commerce infrastructure",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software infrastructure",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software infrastructure",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Interpreters",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Interpreters",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Middleware",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Middleware",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Middleware > Message oriented middleware",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Middleware",
    "node_4": "Message oriented middleware",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Middleware > Reflective middleware",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Middleware",
    "node_4": "Reflective middleware",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Middleware > Embedded middleware",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Middleware",
    "node_4": "Embedded middleware",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Virtual machines",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Virtual machines",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Operating systems",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Operating systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > File systems management",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "File systems management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Memory management",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Memory management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Memory management > Virtual memory",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Memory management",
    "node_4": "Virtual memory",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Memory management > Main memory",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Memory management",
    "node_4": "Main memory",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Memory management > Allocation / deallocation strategies",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Memory management",
    "node_4": "Allocation / deallocation strategies",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Memory management > Garbage collection",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Memory management",
    "node_4": "Garbage collection",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Memory management > Distributed memory",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Memory management",
    "node_4": "Distributed memory",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Memory management > Secondary storage",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Memory management",
    "node_4": "Secondary storage",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Scheduling",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Scheduling",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Deadlocks",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Deadlocks",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Multithreading",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Multithreading",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Multiprocessing / multiprogramming / multitasking",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Multiprocessing / multiprogramming / multitasking",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Monitors",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Monitors",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Mutual exclusion",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Mutual exclusion",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Concurrency control",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Concurrency control",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Power management",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Power management",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Process management > Process synchronization",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Process management",
    "node_4": "Process synchronization",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Communications management",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Communications management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Communications management > Buffering",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Communications management",
    "node_4": "Buffering",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Communications management > Input / output",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Communications management",
    "node_4": "Input / output",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Communications management > Message passing",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Communications management",
    "node_4": "Message passing",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Virtual worlds software",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Virtual worlds software",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Virtual worlds software > Interactive games",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Virtual worlds software",
    "node_4": "Interactive games",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Virtual worlds software > Virtual worlds training simulations",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Virtual worlds software",
    "node_4": "Virtual worlds training simulations",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system structures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system structures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Embedded software",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Embedded software",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > n-tier architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "n-tier architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Peer-to-peer architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Peer-to-peer architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Data flow architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Data flow architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Cooperating communicating processes",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Cooperating communicating processes",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Layered systems",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Layered systems",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Publish-subscribe / event-based architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Publish-subscribe / event-based architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Electronic blackboards",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Electronic blackboards",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Simulator / interpreter",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Simulator / interpreter",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Object oriented architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Object oriented architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Tightly coupled architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Tightly coupled architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > Space-based architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "Space-based architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software architectures > 3-tier architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software architectures",
    "node_4": "3-tier architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system models",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system models > Petri nets",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system models",
    "node_4": "Petri nets",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system models > State systems",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system models",
    "node_4": "State systems",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system models > Entity relationship modeling",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system models",
    "node_4": "Entity relationship modeling",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system models > Model-driven software engineering",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system models",
    "node_4": "Model-driven software engineering",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system models > Feature interaction",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system models",
    "node_4": "Feature interaction",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software system models > Massively parallel systems",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software system models",
    "node_4": "Massively parallel systems",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Ultra-large-scale systems",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Ultra-large-scale systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Distributed systems organizing principles",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Distributed systems organizing principles",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Distributed systems organizing principles > Cloud computing",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Distributed systems organizing principles",
    "node_4": "Cloud computing",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Distributed systems organizing principles > Client-server architectures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Distributed systems organizing principles",
    "node_4": "Client-server architectures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Distributed systems organizing principles > Grid computing",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Distributed systems organizing principles",
    "node_4": "Grid computing",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Distributed systems organizing principles > Organizing principles for web applications",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Distributed systems organizing principles",
    "node_4": "Organizing principles for web applications",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Real-time systems software",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Real-time systems software",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Abstraction, modeling and modularity",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Abstraction, modeling and modularity",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software functional properties",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software functional properties",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Correctness",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Correctness",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Correctness > Synchronization",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Correctness",
    "node_4": "Synchronization",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Correctness > Functionality",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Correctness",
    "node_4": "Functionality",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Correctness > Real-time schedulability",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Correctness",
    "node_4": "Real-time schedulability",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Correctness > Consistency",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Correctness",
    "node_4": "Consistency",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Correctness > Completeness",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Correctness",
    "node_4": "Completeness",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Correctness > Access protection",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Correctness",
    "node_4": "Access protection",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Formal methods",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Formal methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Formal methods > Model checking",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Formal methods",
    "node_4": "Model checking",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Formal methods > Software verification",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Formal methods",
    "node_4": "Software verification",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Formal methods > Automated static analysis",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Formal methods",
    "node_4": "Automated static analysis",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Formal methods > Dynamic analysis",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Formal methods",
    "node_4": "Dynamic analysis",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Extra-functional properties",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Extra-functional properties",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Interoperability",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Interoperability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software performance",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software performance",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software reliability",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software reliability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software fault tolerance",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software fault tolerance",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software fault tolerance > Checkpoint / restart",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software fault tolerance",
    "node_4": "Checkpoint / restart",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software safety",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software safety",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software organization and properties > Software usability",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software organization and properties",
    "node_3": "Software usability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > General programming languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "General programming languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Parallel programming languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Parallel programming languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Distributed programming languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Distributed programming languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Imperative languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Imperative languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Object oriented languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Object oriented languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Functional languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Functional languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Concurrent programming languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Concurrent programming languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Constraint and logic languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Constraint and logic languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Data flow languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Data flow languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Extensible languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Extensible languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Assembly languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Assembly languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Multiparadigm languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Multiparadigm languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language types > Very high level languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language types",
    "node_4": "Very high level languages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Abstract data types",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Abstract data types",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Polymorphism",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Polymorphism",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Inheritance",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Inheritance",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Control structures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Control structures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Data types and structures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Data types and structures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Classes and objects",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Classes and objects",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Modules / packages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Modules / packages",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Constraints",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Constraints",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Recursion",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Recursion",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Concurrent programming structures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Concurrent programming structures",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Procedures, functions and subroutines",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Procedures, functions and subroutines",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Patterns",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Patterns",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Coroutines",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Coroutines",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Language features > Frameworks",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Language features",
    "node_4": "Frameworks",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Formal language definitions",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Formal language definitions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Syntax",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Syntax",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Semantics",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Semantics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Compilers",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Compilers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Interpreters",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Interpreters",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Incremental compilers",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Incremental compilers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Retargetable compilers",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Retargetable compilers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Just-in-time compilers",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Just-in-time compilers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Dynamic compilers",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Dynamic compilers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Translator writing systems and compiler generators",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Translator writing systems and compiler generators",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Source code generation",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Source code generation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Runtime environments",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Runtime environments",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Preprocessors",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Preprocessors",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Parsers",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Parsers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Context specific languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Context specific languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Markup languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Markup languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > State based definitions",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "State based definitions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Visual languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Visual languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Interface definition languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Interface definition languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > System description languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "System description languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Design languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Design languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Unified Modeling Language (UML)",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Unified Modeling Language (UML)",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Architecture description languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Architecture description languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > System modeling languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "System modeling languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Orchestration languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Orchestration languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Integration frameworks",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Integration frameworks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Specification languages",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Specification languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Development frameworks and environments",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Development frameworks and environments",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Object oriented frameworks",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Object oriented frameworks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Software as a service orchestration system",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Software as a service orchestration system",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Integrated and visual development environments",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Integrated and visual development environments",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Application specific development environments",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Application specific development environments",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Software configuration management and version control systems",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Software configuration management and version control systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Software libraries and repositories",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Software libraries and repositories",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software notations and tools > Software maintenance tools",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software notations and tools",
    "node_3": "Software maintenance tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Designing software",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Designing software",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Requirements analysis",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Requirements analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software design engineering",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software design engineering",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software design tradeoffs",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software design tradeoffs",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software implementation planning",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software implementation planning",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software implementation planning > Software design techniques",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software implementation planning",
    "node_4": "Software design techniques",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development process management",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development process management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods > Rapid application development",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "Rapid application development",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods > Agile software development",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "Agile software development",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods > Capability Maturity Model",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "Capability Maturity Model",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods > Waterfall model",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "Waterfall model",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods > Spiral model",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "Spiral model",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods > V-model",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "V-model",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development methods > Design patterns",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development methods",
    "node_4": "Design patterns",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Risk management",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Risk management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software development techniques",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software development techniques",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software prototyping",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software prototyping",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Object oriented development",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Object oriented development",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Flowcharts",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Flowcharts",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Reusability",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Reusability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Reusability > Software product lines",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Reusability",
    "node_4": "Software product lines",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Error handling and recovery",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Error handling and recovery",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Automatic programming",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Automatic programming",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Automatic programming > Genetic programming",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Automatic programming",
    "node_4": "Genetic programming",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software verification and validation",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software verification and validation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Operational analysis",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Operational analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software defect analysis",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software defect analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software defect analysis > Software testing and debugging",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software defect analysis",
    "node_4": "Software testing and debugging",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Fault tree analysis",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Fault tree analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Process validation",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Process validation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Process validation > Walkthroughs",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Process validation",
    "node_4": "Walkthroughs",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Process validation > Pair programming",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Process validation",
    "node_4": "Pair programming",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Process validation > Use cases",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Process validation",
    "node_4": "Use cases",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Process validation > Acceptance testing",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Process validation",
    "node_4": "Acceptance testing",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Process validation > Traceability",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Process validation",
    "node_4": "Traceability",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Formal software verification",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Formal software verification",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Empirical software validation",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Empirical software validation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software post-development issues",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software post-development issues",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software reverse engineering",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software reverse engineering",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Documentation",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Documentation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Backup procedures",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Backup procedures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software evolution",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software evolution",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Software version control",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Software version control",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Maintaining software",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Maintaining software",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > System administration",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "System administration",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Collaboration in software development",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Collaboration in software development",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Open source model",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Open source model",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Programming teams",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Programming teams",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Software and its engineering > Software creation and management > Search-based software engineering",
    "high_level_domain": "Software and its engineering",
    "subdomain": "Software creation and management",
    "node_3": "Search-based software engineering",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Computability",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Computability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Computability > Lambda calculus",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Computability",
    "node_4": "Lambda calculus",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Computability > Turing machines",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Computability",
    "node_4": "Turing machines",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Computability > Recursive functions",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Computability",
    "node_4": "Recursive functions",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Probabilistic computation",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Probabilistic computation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Quantum computation theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Quantum computation theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Quantum computation theory > Quantum complexity theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Quantum computation theory",
    "node_4": "Quantum complexity theory",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Quantum computation theory > Quantum communication complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Quantum computation theory",
    "node_4": "Quantum communication complexity",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Quantum computation theory > Quantum query complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Quantum computation theory",
    "node_4": "Quantum query complexity",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Quantum computation theory > Quantum information theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Quantum computation theory",
    "node_4": "Quantum information theory",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Interactive computation",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Interactive computation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Streaming models",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Streaming models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Concurrency",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Concurrency",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Concurrency > Parallel computing models",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Concurrency",
    "node_4": "Parallel computing models",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Concurrency > Distributed computing models",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Concurrency",
    "node_4": "Distributed computing models",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Concurrency > Process calculi",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Concurrency",
    "node_4": "Process calculi",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Timed and hybrid models",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Timed and hybrid models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Models of computation > Abstract machines",
    "high_level_domain": "Theory of computation",
    "subdomain": "Models of computation",
    "node_3": "Abstract machines",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Formalisms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Formalisms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Formalisms > Algebraic language theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Formalisms",
    "node_4": "Algebraic language theory",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Formalisms > Rewrite systems",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Formalisms",
    "node_4": "Rewrite systems",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Automata over infinite objects",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Automata over infinite objects",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Grammars and context-free languages",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Grammars and context-free languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Tree languages",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Tree languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Automata extensions",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Automata extensions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Automata extensions > Transducers",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Automata extensions",
    "node_4": "Transducers",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Automata extensions > Quantitative automata",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Automata extensions",
    "node_4": "Quantitative automata",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Formal languages and automata theory > Regular languages",
    "high_level_domain": "Theory of computation",
    "subdomain": "Formal languages and automata theory",
    "node_3": "Regular languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Complexity classes",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Complexity classes",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Problems, reductions and completeness",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Problems, reductions and completeness",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Communication complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Communication complexity",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Circuit complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Circuit complexity",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Oracles and decision trees",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Oracles and decision trees",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Algebraic complexity theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Algebraic complexity theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Quantum complexity theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Quantum complexity theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Proof complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Proof complexity",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Interactive proof systems",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Interactive proof systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Complexity theory and logic",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Complexity theory and logic",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Cryptographic primitives",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Cryptographic primitives",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Computational complexity and cryptography > Cryptographic protocols",
    "high_level_domain": "Theory of computation",
    "subdomain": "Computational complexity and cryptography",
    "node_3": "Cryptographic protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Logic and verification",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Logic and verification",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Proof theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Proof theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Modal and temporal logics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Modal and temporal logics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Automated reasoning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Automated reasoning",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Constraint and logic programming",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Constraint and logic programming",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Constructive mathematics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Constructive mathematics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Description logics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Description logics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Equational logic and rewriting",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Equational logic and rewriting",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Finite Model Theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Finite Model Theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Higher order logic",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Higher order logic",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Linear logic",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Linear logic",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Programming logic",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Programming logic",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Abstraction",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Abstraction",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Verification by model checking",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Verification by model checking",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Type theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Type theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Hoare logic",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Hoare logic",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Logic > Separation logic",
    "high_level_domain": "Theory of computation",
    "subdomain": "Logic",
    "node_3": "Separation logic",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Graph algorithms analysis",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Graph algorithms analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Graph algorithms analysis > Network flows",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Graph algorithms analysis",
    "node_4": "Network flows",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Graph algorithms analysis > Sparsification and spanners",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Graph algorithms analysis",
    "node_4": "Sparsification and spanners",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Graph algorithms analysis > Shortest paths",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Graph algorithms analysis",
    "node_4": "Shortest paths",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Graph algorithms analysis > Dynamic graph algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Graph algorithms analysis",
    "node_4": "Dynamic graph algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis > Scheduling algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "Scheduling algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis > Packing and covering problems",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "Packing and covering problems",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis > Routing and network design problems",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "Routing and network design problems",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis > Facility location and clustering",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "Facility location and clustering",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis > Rounding techniques",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "Rounding techniques",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis > Stochastic approximation",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "Stochastic approximation",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Approximation algorithms analysis > Numeric approximation algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Approximation algorithms analysis",
    "node_4": "Numeric approximation algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Discrete optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Discrete optimization",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Discrete optimization > Network optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Discrete optimization",
    "node_5": "Network optimization"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Discrete optimization > Optimization with randomized search heuristics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Discrete optimization",
    "node_5": "Optimization with randomized search heuristics"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Discrete optimization Optimization with randomized search heuristics > Simulated annealing",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Discrete optimization",
    "node_5": "Optimization with randomized search heuristics",
    "node_6": "Simulated annealing"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Discrete optimization Optimization with randomized search heuristics  > Optimization with randomized search heuristics > Evolutionary algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Discrete optimization",
    "node_5": "Optimization with randomized search heuristics",
    "node_6": "Evolutionary algorithms"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Discrete optimization Optimization with randomized search heuristics  > Optimization with randomized search heuristics > Evolutionary algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Discrete optimization",
    "node_5": "Optimization with randomized search heuristics",
    "node_6": "Tabu search"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Discrete optimization Optimization with randomized search heuristics  > Optimization with randomized search heuristics > Evolutionary algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Discrete optimization",
    "node_5": "Optimization with randomized search heuristics",
    "node_6": "Randomized local search"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms >  Mathematical optimization > Continuous optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms >  Mathematical optimization > Continuous optimization > Linear programming",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimizationn",
    "node_4": "Continuous optimization",
    "node_5": "Linear programming"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms >  Mathematical optimization > Continuous optimization > Semidefinite programming",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": "Semidefinite programming"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Continuous optimization > Convex optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": "Convex optimization"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Continuous optimization > Quasiconvex programming and unimodality",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": "Quasiconvex programming and unimodality"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization >  Continuous optimization > Stochastic control and optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": "Stochastic control and optimization"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization >  Continuous optimization > Quadratic programming",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": "Quadratic programming"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Continuous optimization > Nonconvex optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": "Nonconvex optimization"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Continuous optimization > Bio-inspired optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Continuous optimization",
    "node_5": "Bio-inspired optimization"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Mixed discrete-continuous optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Mixed discrete-continuous optimization",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Mixed discrete-continuous optimization > Submodular optimization and polymatroids",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Mixed discrete-continuous optimization",
    "node_5": "Submodular optimization and polymatroids"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Mixed discrete-continuous optimization > Integer programming",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Mixed discrete-continuous optimization",
    "node_5": "Integer programming"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Mixed discrete-continuous optimization > Bio-inspired optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Mixed discrete-continuous optimization",
    "node_5": "Bio-inspired optimization"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Non-parametric optimization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Non-parametric optimization",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Mathematical optimization > Non-parametric optimization > Genetic programming",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Non-parametric optimization",
    "node_5": "Genetic programming"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Non-parametric optimization > Developmental representations",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Mathematical optimization",
    "node_4": "Non-parametric optimization",
    "node_5": "Developmental representations"
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Data structures design and analysis",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Data structures design and analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Data structures design and analysis > Data compression",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Data structures design and analysis",
    "node_4": "Data compression",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Data structures design and analysis > Pattern matching",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Data structures design and analysis",
    "node_4": "Pattern matching",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Data structures design and analysis > Sorting and searching",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Data structures design and analysis",
    "node_4": "Sorting and searching",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Data structures design and analysis > Predecessor queries",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Data structures design and analysis",
    "node_4": "Predecessor queries",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Data structures design and analysis > Cell probe models and lower bounds",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Data structures design and analysis",
    "node_4": "Cell probe models and lower bounds",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Online algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Online algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Online learning algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Online learning algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Online learning algorithms > Scheduling algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Online learning algorithms",
    "node_4": "Scheduling algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parameterized complexity and exact algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parameterized complexity and exact algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parameterized complexity and exact algorithms > Fixed parameter tractability",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parameterized complexity and exact algorithms",
    "node_4": "Fixed parameter tractability",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parameterized complexity and exact algorithms > W hierarchy",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parameterized complexity and exact algorithms",
    "node_4": "W hierarchy",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Streaming, sublinear and near linear time algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Streaming, sublinear and near linear time algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Streaming, sublinear and near linear time algorithms > Bloom filters and hashing",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Streaming, sublinear and near linear time algorithms",
    "node_4": "Bloom filters and hashing",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Streaming, sublinear and near linear time algorithms > Sketching and sampling",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Streaming, sublinear and near linear time algorithms",
    "node_4": "Sketching and sampling",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Streaming, sublinear and near linear time algorithms > Lower bounds and information complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Streaming, sublinear and near linear time algorithms",
    "node_4": "Lower bounds and information complexity",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Streaming, sublinear and near linear time algorithms > Random order and robust communication complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Streaming, sublinear and near linear time algorithms",
    "node_4": "Random order and robust communication complexity",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Streaming, sublinear and near linear time algorithms > Nearest neighbor algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Streaming, sublinear and near linear time algorithms",
    "node_4": "Nearest neighbor algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parallel algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parallel algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parallel algorithms > MapReduce algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parallel algorithms",
    "node_4": "MapReduce algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parallel algorithms > Self-organization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parallel algorithms",
    "node_4": "Self-organization",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parallel algorithms > Shared memory algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parallel algorithms",
    "node_4": "Shared memory algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parallel algorithms > Vector / streaming algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parallel algorithms",
    "node_4": "Vector / streaming algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Parallel algorithms > Massively parallel algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Parallel algorithms",
    "node_4": "Massively parallel algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Distributed algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Distributed algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Distributed algorithms > MapReduce algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Distributed algorithms",
    "node_4": "MapReduce algorithms",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Distributed algorithms > Self-organization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Distributed algorithms",
    "node_4": "Self-organization",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Algorithm design techniques",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Algorithm design techniques",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Algorithm design techniques > Backtracking",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Algorithm design techniques",
    "node_4": "Backtracking",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Algorithm design techniques > Branch-and-bound",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Algorithm design techniques",
    "node_4": "Branch-and-bound",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Algorithm design techniques > Divide and conquer",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Algorithm design techniques",
    "node_4": "Divide and conquer",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Algorithm design techniques > Dynamic programming",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Algorithm design techniques",
    "node_4": "Dynamic programming",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Algorithm design techniques > Preconditioning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Algorithm design techniques",
    "node_4": "Preconditioning",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Design and analysis of algorithms > Concurrent algorithms",
    "high_level_domain": "Theory of computation",
    "subdomain": "Design and analysis of algorithms",
    "node_3": "Concurrent algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Pseudorandomness and derandomization",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Pseudorandomness and derandomization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Computational geometry",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Computational geometry",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Generating random combinatorial structures",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Generating random combinatorial structures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Random walks and Markov chains",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Random walks and Markov chains",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Expander graphs and randomness extractors",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Expander graphs and randomness extractors",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Error-correcting codes",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Error-correcting codes",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Random projections and metric embeddings",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Random projections and metric embeddings",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Random network models",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Random network models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Randomness, geometry and discrete structures > Random search heuristics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Randomness, geometry and discrete structures",
    "node_3": "Random search heuristics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Machine learning theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Machine learning theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Kernel methods",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Kernel methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Kernel methods > Support vector machines",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Kernel methods",
    "node_4": "Support vector machines",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Kernel methods > Gaussian processes",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Kernel methods",
    "node_4": "Gaussian processes",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Multi-agent learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Multi-agent learning",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Multi-agent learning > Models of learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Multi-agent learning",
    "node_4": "Models of learning",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Multi-agent learning > Query learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Multi-agent learning",
    "node_4": "Query learning",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Multi-agent learning > Structured prediction",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Multi-agent learning",
    "node_4": "Structured prediction",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Reinforcement learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Reinforcement learning",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Reinforcement learning > Sequential decision making",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Reinforcement learning",
    "node_4": "Sequential decision making",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Reinforcement learning > Inverse reinforcement learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Reinforcement learning",
    "node_4": "Inverse reinforcement learning",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Reinforcement learning > Apprenticeship learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Reinforcement learning",
    "node_4": "Apprenticeship learning",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Reinforcement learning > Multi-agent reinforcement learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Reinforcement learning",
    "node_4": "Multi-agent reinforcement learning",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Reinforcement learning > Adversarial learning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Reinforcement learning",
    "node_4": "Adversarial learning",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Algorithmic game theory and mechanism design",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Algorithmic game theory and mechanism design",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Solution concepts in game theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Solution concepts in game theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Solution concepts in game theory > Exact and approximate computation of equilibria",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Solution concepts in game theory",
    "node_4": "Exact and approximate computation of equilibria",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Solution concepts in game theory > Quality of equilibria",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Solution concepts in game theory",
    "node_4": "Quality of equilibria",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Solution concepts in game theory > Convergence and learning in games",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Solution concepts in game theory",
    "node_4": "Convergence and learning in games",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Market equilibria",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Market equilibria",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Market equilibria > Computational pricing and auctions",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Market equilibria",
    "node_4": "Computational pricing and auctions",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Market equilibria > Representations of games and their complexity",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Market equilibria",
    "node_4": "Representations of games and their complexity",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Market equilibria > Network games",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Market equilibria",
    "node_4": "Network games",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Market equilibria > Network formation",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Market equilibria",
    "node_4": "Network formation",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Market equilibria > Computational advertising theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Market equilibria",
    "node_4": "Computational advertising theory",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Data exchange",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Data exchange",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Data provenance",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Data provenance",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Data modeling",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Data modeling",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Database query languages (principles)",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Database query languages (principles)",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Database constraints theory",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Database constraints theory",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Database interoperability",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Database interoperability",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Data structures and algorithms for data management",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Data structures and algorithms for data management",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Database query processing and optimization (theory)",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Database query processing and optimization (theory)",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Data integration",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Data integration",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Logic and databases",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Logic and databases",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Theory of database privacy and security",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Theory of database privacy and security",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Database theory > Incomplete, inconsistent, and uncertain databases",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Database theory",
    "node_4": "Incomplete, inconsistent, and uncertain databases",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Theory and algorithms for application domains > Theory of randomized search heuristics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Theory and algorithms for application domains",
    "node_3": "Theory of randomized search heuristics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program constructs",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program constructs",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program constructs > Control primitives",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program constructs",
    "node_4": "Control primitives",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program constructs > Functional constructs",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program constructs",
    "node_4": "Functional constructs",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program constructs > Object oriented constructs",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program constructs",
    "node_4": "Object oriented constructs",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program constructs > Program schemes",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program constructs",
    "node_4": "Program schemes",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program constructs > Type structures",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program constructs",
    "node_4": "Type structures",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program semantics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program semantics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program semantics > Algebraic semantics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program semantics",
    "node_4": "Algebraic semantics",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program semantics > Denotational semantics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program semantics",
    "node_4": "Denotational semantics",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program semantics > Operational semantics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program semantics",
    "node_4": "Operational semantics",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program semantics > Axiomatic semantics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program semantics",
    "node_4": "Axiomatic semantics",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program semantics > Action semantics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program semantics",
    "node_4": "Action semantics",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program semantics > Categorical semantics",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program semantics",
    "node_4": "Categorical semantics",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Invariants",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Invariants",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Program specifications",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Program specifications",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Pre- and post-conditions",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Pre- and post-conditions",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Program verification",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Program verification",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Program analysis",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Program analysis",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Assertions",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Assertions",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Parsing",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Parsing",
    "node_5": ""
  },
  {
    "path": "Theory of computation > Semantics and reasoning > Program reasoning > Abstraction",
    "high_level_domain": "Theory of computation",
    "subdomain": "Semantics and reasoning",
    "node_3": "Program reasoning",
    "node_4": "Abstraction",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics > Combinatoric problems",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "Combinatoric problems",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics > Permutations and combinations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "Permutations and combinations",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics > Combinatorial algorithms",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "Combinatorial algorithms",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics > Generating functions",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "Generating functions",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics > Combinatorial optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "Combinatorial optimization",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics > Combinatorics on words",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "Combinatorics on words",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Combinatorics > Enumeration",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Combinatorics",
    "node_4": "Enumeration",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Trees",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Trees",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Hypergraphs",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Hypergraphs",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Random graphs",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Random graphs",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Graph coloring",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Graph coloring",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Paths and connectivity problems",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Paths and connectivity problems",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Graph enumeration",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Graph enumeration",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Matchings and factors",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Matchings and factors",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Graphs and surfaces",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Graphs and surfaces",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Network flows",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Network flows",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Spectra of graphs",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Spectra of graphs",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Extremal graph theory",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Extremal graph theory",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Matroids and greedoids",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Matroids and greedoids",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Graph algorithms",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Graph algorithms",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Discrete mathematics > Graph theory > Approximation algorithms",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Discrete mathematics",
    "node_3": "Graph theory",
    "node_4": "Approximation algorithms",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Probabilistic representations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Probabilistic representations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Equational models",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Equational models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Equational models > Causal networks",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Equational models",
    "node_4": "Causal networks",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Equational models > Stochastic differential equations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Equational models",
    "node_4": "Stochastic differential equations",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Nonparametric representations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Nonparametric representations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Nonparametric representations > Kernel density estimators",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Nonparametric representations",
    "node_4": "Kernel density estimators",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Nonparametric representations > Spline models",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Nonparametric representations",
    "node_4": "Spline models",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Nonparametric representations > Bayesian nonparametric models",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Nonparametric representations",
    "node_4": "Bayesian nonparametric models",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Probabilistic inference problems",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Probabilistic inference problems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Density estimation",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Density estimation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Density estimation > Quantile regression",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Density estimation",
    "node_4": "Quantile regression",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Probabilistic reasoning algorithms",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Probabilistic reasoning algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Markov-chain Monte Carlo methods",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Markov-chain Monte Carlo methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Markov-chain Monte Carlo methods > Gibbs sampling",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Markov-chain Monte Carlo methods",
    "node_4": "Gibbs sampling",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Markov-chain Monte Carlo methods > Metropolis-Hastings algorithm",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Markov-chain Monte Carlo methods",
    "node_4": "Metropolis-Hastings algorithm",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Markov-chain Monte Carlo methods > Simulated annealing",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Markov-chain Monte Carlo methods",
    "node_4": "Simulated annealing",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Markov-chain Monte Carlo methods > Markov-chain Monte Carlo convergence measures",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Markov-chain Monte Carlo methods",
    "node_4": "Markov-chain Monte Carlo convergence measures",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Markov-chain Monte Carlo methods > Sequential Monte Carlo methods",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Markov-chain Monte Carlo methods",
    "node_4": "Sequential Monte Carlo methods",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Kalman filters and hidden Markov models",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Kalman filters and hidden Markov models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Resampling methods",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Resampling methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Resampling methods > Bootstrapping",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Resampling methods",
    "node_4": "Bootstrapping",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Resampling methods > Jackknifing",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Resampling methods",
    "node_4": "Jackknifing",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Probabilistic algorithms",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Probabilistic algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Statistical paradigms",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Statistical paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Regression analysis",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Regression analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Regression analysis > Robust regression",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Regression analysis",
    "node_4": "Robust regression",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Stochastic processes",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Stochastic processes",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Stochastic processes > Markov processes",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Stochastic processes",
    "node_4": "Markov processes",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Nonparametric statistics",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Nonparametric statistics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Distribution functions",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Distribution functions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Probability and statistics > Multivariate statistics",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Probability and statistics",
    "node_3": "Multivariate statistics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical software > Solvers",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical software",
    "node_3": "Solvers",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical software > Statistical software",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical software",
    "node_3": "Statistical software",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical software > Mathematical software performance",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical software",
    "node_3": "Mathematical software performance",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Information theory > Coding theory",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Information theory",
    "node_3": "Coding theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Numerical analysis",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Numerical analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Computations on polynomials",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Computations on polynomials",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Computations on polynomials > Gröbner bases and other special bases",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Computations on polynomials",
    "node_4": "Gröbner bases and other special bases",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Mathematical optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Mathematical optimization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Discrete optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Discrete optimization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Optimization with randomized search heuristics",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Optimization with randomized search heuristics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Optimization with randomized search heuristics > Simulated annealing",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Optimization with randomized search heuristics",
    "node_4": "Simulated annealing",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Optimization with randomized search heuristics > Evolutionary algorithms",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Optimization with randomized search heuristics",
    "node_4": "Evolutionary algorithms",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Optimization with randomized search heuristics > Tabu search",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Optimization with randomized search heuristics",
    "node_4": "Tabu search",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Optimization with randomized search heuristics > Randomized local search",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Optimization with randomized search heuristics",
    "node_4": "Randomized local search",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Linear programming",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Linear programming",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Semidefinite programming",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Semidefinite programming",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Convex optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Convex optimization",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Quasiconvex programming and unimodality",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Quasiconvex programming and unimodality",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Stochastic control and optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Stochastic control and optimization",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Quadratic programming",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Quadratic programming",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Nonconvex optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Nonconvex optimization",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Continuous optimization > Bio-inspired optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Continuous optimization",
    "node_4": "Bio-inspired optimization",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Mixed discrete-continuous optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Mixed discrete-continuous optimization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Mixed discrete-continuous optimization > Submodular optimization and polymatroids",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Mixed discrete-continuous optimization",
    "node_4": "Submodular optimization and polymatroids",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Mixed discrete-continuous optimization > Integer programming",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Mixed discrete-continuous optimization",
    "node_4": "Integer programming",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Mixed discrete-continuous optimization > Bio-inspired optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Mixed discrete-continuous optimization",
    "node_4": "Bio-inspired optimization",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Non-parametric optimization",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Non-parametric optimization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Non-parametric optimization > Genetic programming",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Non-parametric optimization",
    "node_4": "Genetic programming",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Non-parametric optimization > Developmental representations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Non-parametric optimization",
    "node_4": "Developmental representations",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Differential equations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Differential equations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Differential equations > Ordinary differential equations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Differential equations",
    "node_4": "Ordinary differential equations",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Differential equations > Partial differential equations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Differential equations",
    "node_4": "Partial differential equations",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Differential equations > Differential algebraic equations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Differential equations",
    "node_4": "Differential algebraic equations",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Differential equations > Differential variational inequalities",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Differential equations",
    "node_4": "Differential variational inequalities",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Calculus",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Calculus > Lambda calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Calculus",
    "node_4": "Lambda calculus",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Calculus > Differential calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Calculus",
    "node_4": "Differential calculus",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Calculus > Integral calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Calculus",
    "node_4": "Integral calculus",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Functional analysis",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Functional analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Functional analysis > Approximation",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Functional analysis",
    "node_4": "Approximation",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Integral equations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Integral equations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Nonlinear equations",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Nonlinear equations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Mathematical analysis > Quadrature",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Mathematical analysis",
    "node_3": "Quadrature",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Calculus",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Calculus > Lambda calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Calculus",
    "node_4": "Lambda calculus",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Calculus > Differential calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Calculus",
    "node_4": "Differential calculus",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Calculus > Integral calculus",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Calculus",
    "node_4": "Integral calculus",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Topology",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Topology",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Topology > Point-set topology",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Topology",
    "node_4": "Point-set topology",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Topology > Algebraic topology",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Topology",
    "node_4": "Algebraic topology",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Topology > Geometric topology",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Topology",
    "node_4": "Geometric topology",
    "node_5": ""
  },
  {
    "path": "Mathematics of computing > Continuous mathematics > Continuous functions",
    "high_level_domain": "Mathematics of computing",
    "subdomain": "Continuous mathematics",
    "node_3": "Continuous functions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database design and models",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database design and models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Graph-based database models",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Graph-based database models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Graph-based database models > Hierarchical data models",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Graph-based database models",
    "node_4": "Hierarchical data models",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Graph-based database models > Network data models",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Graph-based database models",
    "node_4": "Network data models",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions > Semi-structured data",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "Semi-structured data",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions > Data streams",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "Data streams",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions > Data provenance",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "Data provenance",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions > Incomplete data",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "Incomplete data",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions > Temporal data",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "Temporal data",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions > Uncertainty",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "Uncertainty",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data model extensions > Inconsistent data",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data model extensions",
    "node_4": "Inconsistent data",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data structures",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data structures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data access methods",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data access methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data access methods > Multidimensional range search",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data access methods",
    "node_4": "Multidimensional range search",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data access methods > Data scans",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data access methods",
    "node_4": "Data scans",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data access methods > Point lookups",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data access methods",
    "node_4": "Point lookups",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data access methods > Unidimensional range search",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data access methods",
    "node_4": "Unidimensional range search",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data access methods > Proximity search",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data access methods",
    "node_4": "Proximity search",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data layout",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data layout",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data layout > Data compression",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data layout",
    "node_4": "Data compression",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data layout > Data encryption",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data layout",
    "node_4": "Data encryption",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Data layout > Record and block layout",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Data layout",
    "node_4": "Record and block layout",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database management system engines",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database management system engines",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database query processing",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database query processing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database query processing > Query optimization",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database query processing",
    "node_4": "Query optimization",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database query processing > Query operators",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database query processing",
    "node_4": "Query operators",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database query processing > Query planning",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database query processing",
    "node_4": "Query planning",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database query processing > Join algorithms",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database query processing",
    "node_4": "Join algorithms",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database transaction processing",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database transaction processing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database transaction processing > Data locking",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database transaction processing",
    "node_4": "Data locking",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database transaction processing > Transaction logging",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database transaction processing",
    "node_4": "Transaction logging",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database transaction processing > Database recovery",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database transaction processing",
    "node_4": "Database recovery",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Parallel and distributed DBMSs",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Parallel and distributed DBMSs",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Parallel and distributed DBMSs > Key-value stores",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Parallel and distributed DBMSs",
    "node_4": "Key-value stores",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Parallel and distributed DBMSs > MapReduce-based systems",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Parallel and distributed DBMSs",
    "node_4": "MapReduce-based systems",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Parallel and distributed DBMSs > Relational parallel and distributed DBMSs",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Parallel and distributed DBMSs",
    "node_4": "Relational parallel and distributed DBMSs",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Distributed database transactions",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Distributed database transactions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Distributed database transactions > Distributed data locking",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Distributed database transactions",
    "node_4": "Distributed data locking",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Distributed database transactions > Deadlocks",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Distributed database transactions",
    "node_4": "Deadlocks",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Distributed database transactions > Distributed database recovery",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Distributed database transactions",
    "node_4": "Distributed database recovery",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Query languages",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Query languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Relational database query languages",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Relational database query languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Relational database query languages > Structured Query Language",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Relational database query languages",
    "node_4": "Structured Query Language",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > XML query languages",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "XML query languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > XML query languages > XPath",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "XML query languages",
    "node_4": "XPath",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > XML query languages > XQuery",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "XML query languages",
    "node_4": "XQuery",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Query languages for non-relational engines",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Query languages for non-relational engines",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Query languages for non-relational engines > MapReduce languages",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Query languages for non-relational engines",
    "node_4": "MapReduce languages",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database administration",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database administration",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database administration > Database utilities and tools",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database administration",
    "node_4": "Database utilities and tools",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database administration > Database performance evaluation",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database administration",
    "node_4": "Database performance evaluation",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database administration > Autonomous database administration",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database administration",
    "node_4": "Autonomous database administration",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Database administration > Data dictionaries",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Database administration",
    "node_4": "Data dictionaries",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Deduplication",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Deduplication",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Extraction, transformation and loading",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Extraction, transformation and loading",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Data exchange",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Data exchange",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Data cleaning",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Data cleaning",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Wrappers (data mining)",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Wrappers (data mining)",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Mediators and data integration",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Mediators and data integration",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Entity resolution",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Entity resolution",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Data warehouses",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Data warehouses",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Information integration > Federated databases",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Information integration",
    "node_4": "Federated databases",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Database web servers",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Database web servers",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Application servers",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Application servers",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Object-relational mapping facilities",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Object-relational mapping facilities",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Data federation tools",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Data federation tools",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Data replication tools",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Data replication tools",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Distributed transaction monitors",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Distributed transaction monitors",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Message queues",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Message queues",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Service buses",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Service buses",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Enterprise application integration tools",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Enterprise application integration tools",
    "node_5": ""
  },
  {
    "path": "Information systems > Data management systems > Middleware for databases > Middleware business process managers",
    "high_level_domain": "Information systems",
    "subdomain": "Data management systems",
    "node_3": "Middleware for databases",
    "node_4": "Middleware business process managers",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Information storage technologies",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Information storage technologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage class memory",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage class memory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage class memory > Flash memory",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage class memory",
    "node_4": "Flash memory",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage class memory > Phase change memory",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage class memory",
    "node_4": "Phase change memory",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Record storage systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Record storage systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Record storage alternatives",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Record storage alternatives",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Directory structures",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Directory structures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Directory structures > B-trees",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Directory structures",
    "node_4": "B-trees",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Directory structures > Vnodes",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Directory structures",
    "node_4": "Vnodes",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Directory structures > Inodes",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Directory structures",
    "node_4": "Inodes",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Directory structures > Extent-based file structures",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Directory structures",
    "node_4": "Extent-based file structures",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Block / page strategies",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Block / page strategies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Block / page strategies > Slotted pages",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Block / page strategies",
    "node_4": "Slotted pages",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Block / page strategies > Intrapage space management",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Block / page strategies",
    "node_4": "Intrapage space management",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Block / page strategies > Interpage free-space management",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Block / page strategies",
    "node_4": "Interpage free-space management",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Record layout alternatives",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Record layout alternatives",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Record layout alternatives > Fixed length attributes",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Record layout alternatives",
    "node_4": "Fixed length attributes",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Record layout alternatives > Variable length attributes",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Record layout alternatives",
    "node_4": "Variable length attributes",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Record layout alternatives > Null values in records",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Record layout alternatives",
    "node_4": "Null values in records",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Relational storage",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Relational storage",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Relational storage > Horizontal partitioning",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Relational storage",
    "node_4": "Horizontal partitioning",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Relational storage > Vertical partitioning",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Relational storage",
    "node_4": "Vertical partitioning",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Relational storage > Column based storage",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Relational storage",
    "node_4": "Column based storage",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Relational storage > Hybrid storage layouts",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Relational storage",
    "node_4": "Hybrid storage layouts",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Relational storage > Compression strategies",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Relational storage",
    "node_4": "Compression strategies",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage replication",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage replication",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage replication > Mirroring",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage replication",
    "node_4": "Mirroring",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage replication > RAID",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage replication",
    "node_4": "RAID",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage replication > Point-in-time copies",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage replication",
    "node_4": "Point-in-time copies",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage replication > Remote replication",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage replication",
    "node_4": "Remote replication",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage replication > Storage recovery strategies",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage replication",
    "node_4": "Storage recovery strategies",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage architectures",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage network architectures",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage network architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage network architectures > Storage area networks",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage network architectures",
    "node_4": "Storage area networks",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage network architectures > Direct attached storage",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage network architectures",
    "node_4": "Direct attached storage",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage network architectures > Network attached storage",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage network architectures",
    "node_4": "Network attached storage",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage management",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage management > Hierarchical storage management",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage management",
    "node_4": "Hierarchical storage management",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage management > Storage virtualization",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage management",
    "node_4": "Storage virtualization",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage management > Information lifecycle management",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage management",
    "node_4": "Information lifecycle management",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage management > Version management",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage management",
    "node_4": "Version management",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage management > Storage power management",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage management",
    "node_4": "Storage power management",
    "node_5": ""
  },
  {
    "path": "Information systems > Information storage systems > Storage management > Thin provisioning",
    "high_level_domain": "Information systems",
    "subdomain": "Information storage systems",
    "node_3": "Storage management",
    "node_4": "Thin provisioning",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Enterprise information systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Enterprise information systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Enterprise information systems > Intranets",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Enterprise information systems",
    "node_4": "Intranets",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Enterprise information systems > Extranets",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Enterprise information systems",
    "node_4": "Extranets",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Enterprise information systems > Enterprise resource planning",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Enterprise information systems",
    "node_4": "Enterprise resource planning",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Enterprise information systems > Enterprise applications",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Enterprise information systems",
    "node_4": "Enterprise applications",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Enterprise information systems > Data centers",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Enterprise information systems",
    "node_4": "Data centers",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Blogs",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Blogs",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Wikis",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Wikis",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Reputation systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Reputation systems",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Open source software",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Open source software",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Social networking sites",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Social networking sites",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Social tagging systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Social tagging systems",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Synchronous editors",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Synchronous editors",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Collaborative and social computing systems and tools > Asynchronous editors",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Asynchronous editors",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Spatial-temporal systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Spatial-temporal systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Spatial-temporal systems > Location based services",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Spatial-temporal systems",
    "node_4": "Location based services",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Spatial-temporal systems > Geographic information systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Spatial-temporal systems",
    "node_4": "Geographic information systems",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Spatial-temporal systems > Sensor networks",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Spatial-temporal systems",
    "node_4": "Sensor networks",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Spatial-temporal systems > Data streaming",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Spatial-temporal systems",
    "node_4": "Data streaming",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Spatial-temporal systems > Global positioning systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Spatial-temporal systems",
    "node_4": "Global positioning systems",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Decision support systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Decision support systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Decision support systems > Data warehouses",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Decision support systems",
    "node_4": "Data warehouses",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Decision support systems > Expert systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Decision support systems",
    "node_4": "Expert systems",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Decision support systems > Data analytics",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Decision support systems",
    "node_4": "Data analytics",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Decision support systems > Online analytical processing",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Decision support systems",
    "node_4": "Online analytical processing",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Mobile information processing systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Mobile information processing systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Process control systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Process control systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Multimedia information systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Multimedia information systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Multimedia information systems > Multimedia databases",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Multimedia information systems",
    "node_4": "Multimedia databases",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Multimedia information systems > Multimedia streaming",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Multimedia information systems",
    "node_4": "Multimedia streaming",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Multimedia information systems > Multimedia content creation",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Multimedia information systems",
    "node_4": "Multimedia content creation",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Multimedia information systems > Massively multiplayer online games",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Multimedia information systems",
    "node_4": "Massively multiplayer online games",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Data mining",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Data mining",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Data mining > Data cleaning",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Data mining",
    "node_4": "Data cleaning",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Data mining > Collaborative filtering",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Data mining",
    "node_4": "Collaborative filtering",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Data mining > Association rules",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Data mining",
    "node_4": "Association rules",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Data mining > Clustering",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Data mining",
    "node_4": "Clustering",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Data mining > Nearest-neighbor search",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Data mining",
    "node_4": "Nearest-neighbor search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Data mining > Data stream mining",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Data mining",
    "node_4": "Data stream mining",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Digital libraries and archives",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Digital libraries and archives",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Computational advertising",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Computational advertising",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information systems applications > Computing platforms",
    "high_level_domain": "Information systems",
    "subdomain": "Information systems applications",
    "node_3": "Computing platforms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web searching and information discovery",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web searching and information discovery",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web search engines",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web search engines",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web search engines > Web crawling",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web search engines",
    "node_4": "Web crawling",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web search engines > Web indexing",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web search engines",
    "node_4": "Web indexing",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web search engines > Page and site ranking",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web search engines",
    "node_4": "Page and site ranking",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web search engines > Spam detection",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web search engines",
    "node_4": "Spam detection",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Online advertising",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Online advertising",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Online advertising > Sponsored search advertising",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Online advertising",
    "node_4": "Sponsored search advertising",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Online advertising > Content match advertising",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Online advertising",
    "node_4": "Content match advertising",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Online advertising > Display advertising",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Online advertising",
    "node_4": "Display advertising",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Online advertising > Social advertising",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Online advertising",
    "node_4": "Social advertising",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web mining",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web mining",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Data extraction and integration",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Data extraction and integration",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Data extraction and integration > Deep web",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Data extraction and integration",
    "node_4": "Deep web",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Data extraction and integration > Surfacing",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Data extraction and integration",
    "node_4": "Surfacing",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Data extraction and integration > Search results deduplication",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Data extraction and integration",
    "node_4": "Search results deduplication",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web applications",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web applications",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Internet communications tools",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Internet communications tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Internet communications tools > Email",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Internet communications tools",
    "node_4": "Email",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Internet communications tools > Blogs",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Internet communications tools",
    "node_4": "Blogs",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Internet communications tools > Texting",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Internet communications tools",
    "node_4": "Texting",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Internet communications tools > Chat",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Internet communications tools",
    "node_4": "Chat",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Internet communications tools > Web conferencing",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Internet communications tools",
    "node_4": "Web conferencing",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Crowdsourcing",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Crowdsourcing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Crowdsourcing > Answer ranking",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Crowdsourcing",
    "node_4": "Answer ranking",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Crowdsourcing > Trust",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Crowdsourcing",
    "node_4": "Trust",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Crowdsourcing > Incentive schemes",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Crowdsourcing",
    "node_4": "Incentive schemes",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Crowdsourcing > Reputation systems",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Crowdsourcing",
    "node_4": "Reputation systems",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > Digital cash",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "Digital cash",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > E-commerce infrastructure",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "E-commerce infrastructure",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > Electronic data interchange",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "Electronic data interchange",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > Electronic funds transfer",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "Electronic funds transfer",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > Online shopping",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "Online shopping",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > Online banking",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "Online banking",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > Secure online transactions",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "Secure online transactions",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Electronic commerce > Online auctions",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Electronic commerce",
    "node_4": "Online auctions",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web interfaces",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web interfaces",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web interfaces > Wikis",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web interfaces",
    "node_4": "Wikis",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web interfaces > Browsers",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web interfaces",
    "node_4": "Browsers",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web interfaces > Mashups",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web interfaces",
    "node_4": "Mashups",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web services",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web services",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web services > Simple Object Access Protocol (SOAP)",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web services",
    "node_4": "Simple Object Access Protocol (SOAP)",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web services > RESTful web services",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web services",
    "node_4": "RESTful web services",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web services > Web Services Description Language (WSDL)",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web services",
    "node_4": "Web Services Description Language (WSDL)",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web services > Universal Description Discovery and Integration (UDDI)",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web services",
    "node_4": "Universal Description Discovery and Integration (UDDI)",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web services > Service discovery and interfaces",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web services",
    "node_4": "Service discovery and interfaces",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Web data description languages",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Web data description languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Semantic web description languages",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Semantic web description languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Semantic web description languages > Resource Description Framework (RDF)",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Semantic web description languages",
    "node_4": "Resource Description Framework (RDF)",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Semantic web description languages > Web Ontology Language (OWL)",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Semantic web description languages",
    "node_4": "Web Ontology Language (OWL)",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Markup languages",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Markup languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Markup languages > Extensible Markup Language (XML)",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Markup languages",
    "node_4": "Extensible Markup Language (XML)",
    "node_5": ""
  },
  {
    "path": "Information systems > World Wide Web > Markup languages > Hypertext languages",
    "high_level_domain": "Information systems",
    "subdomain": "World Wide Web",
    "node_3": "Markup languages",
    "node_4": "Hypertext languages",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Document structure",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Document structure",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Document topic models",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Document topic models",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Content analysis and feature selection",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Content analysis and feature selection",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Data encoding and canonicalization",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Data encoding and canonicalization",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Document collection models",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Document collection models",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Ontologies",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Ontologies",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Dictionaries",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Dictionaries",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Document representation > Thesauri",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Document representation",
    "node_4": "Thesauri",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Information retrieval query processing",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Information retrieval query processing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Information retrieval query processing > Query representation",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Information retrieval query processing",
    "node_4": "Query representation",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Information retrieval query processing > Query intent",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Information retrieval query processing",
    "node_4": "Query intent",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Information retrieval query processing > Query log analysis",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Information retrieval query processing",
    "node_4": "Query log analysis",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Information retrieval query processing > Query suggestion",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Information retrieval query processing",
    "node_4": "Query suggestion",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Information retrieval query processing > Query reformulation",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Information retrieval query processing",
    "node_4": "Query reformulation",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Users and interactive retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Users and interactive retrieval",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Users and interactive retrieval > Personalization",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Users and interactive retrieval",
    "node_4": "Personalization",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Users and interactive retrieval > Task models",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Users and interactive retrieval",
    "node_4": "Task models",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Users and interactive retrieval > Search interfaces",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Users and interactive retrieval",
    "node_4": "Search interfaces",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Users and interactive retrieval > Collaborative search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Users and interactive retrieval",
    "node_4": "Collaborative search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Rank aggregation",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Rank aggregation",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Probabilistic retrieval models",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Probabilistic retrieval models",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Language models",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Language models",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Similarity measures",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Similarity measures",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Learning to rank",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Learning to rank",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Combination, fusion and federated search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Combination, fusion and federated search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Information retrieval diversity",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Information retrieval diversity",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Top-k retrieval in databases",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Top-k retrieval in databases",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval models and ranking > Novelty in information retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval models and ranking",
    "node_4": "Novelty in information retrieval",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Question answering",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Question answering",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Document filtering",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Document filtering",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Recommender systems",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Recommender systems",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Information extraction",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Information extraction",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Sentiment analysis",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Sentiment analysis",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Expert search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Expert search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Near-duplicate and plagiarism detection",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Near-duplicate and plagiarism detection",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Clustering and classification",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Clustering and classification",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Summarization",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Summarization",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Retrieval tasks and goals > Business intelligence",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Retrieval tasks and goals",
    "node_4": "Business intelligence",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Evaluation of retrieval results",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Evaluation of retrieval results",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Evaluation of retrieval results > Test collections",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Evaluation of retrieval results",
    "node_4": "Test collections",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Evaluation of retrieval results > Relevance assessment",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Evaluation of retrieval results",
    "node_4": "Relevance assessment",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Evaluation of retrieval results > Retrieval effectiveness",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Evaluation of retrieval results",
    "node_4": "Retrieval effectiveness",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Evaluation of retrieval results > Retrieval efficiency",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Evaluation of retrieval results",
    "node_4": "Retrieval efficiency",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Evaluation of retrieval results > Presentation of retrieval results",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Evaluation of retrieval results",
    "node_4": "Presentation of retrieval results",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Search engine indexing",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Search engine indexing",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Search index compression",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Search index compression",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Distributed retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Distributed retrieval",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Peer-to-peer retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Peer-to-peer retrieval",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Retrieval on mobile devices",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Retrieval on mobile devices",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Adversarial retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Adversarial retrieval",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Link and co-citation analysis",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Link and co-citation analysis",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Search engine architectures and scalability > Searching with auxiliary databases",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Search engine architectures and scalability",
    "node_4": "Searching with auxiliary databases",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Specialized information retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Specialized information retrieval",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Specialized information retrieval > Structure and multilingual text search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Specialized information retrieval",
    "node_4": "Structure and multilingual text search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Specialized information retrieval > Structured text search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Specialized information retrieval",
    "node_4": "Structured text search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Specialized information retrieval > Mathematics retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Specialized information retrieval",
    "node_4": "Mathematics retrieval",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Specialized information retrieval > Chemical and biochemical retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Specialized information retrieval",
    "node_4": "Chemical and biochemical retrieval",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Specialized information retrieval > Multilingual and cross-lingual retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Specialized information retrieval",
    "node_4": "Multilingual and cross-lingual retrieval",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Multimedia and multimodal retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Multimedia and multimodal retrieval",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Environment-specific retrieval",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Environment-specific retrieval",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Environment-specific retrieval > Enterprise search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Environment-specific retrieval",
    "node_4": "Enterprise search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Environment-specific retrieval > Desktop search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Environment-specific retrieval",
    "node_4": "Desktop search",
    "node_5": ""
  },
  {
    "path": "Information systems > Information retrieval > Environment-specific retrieval > Web and social media search",
    "high_level_domain": "Information systems",
    "subdomain": "Information retrieval",
    "node_3": "Environment-specific retrieval",
    "node_4": "Web and social media search",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Key management",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Key management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Public key (asymmetric) techniques",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Public key (asymmetric) techniques",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Public key (asymmetric) techniques > Digital signatures",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Public key (asymmetric) techniques",
    "node_4": "Digital signatures",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Public key (asymmetric) techniques > Public key encryption",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Public key (asymmetric) techniques",
    "node_4": "Public key encryption",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Symmetric cryptography and hash functions",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Symmetric cryptography and hash functions",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Symmetric cryptography and hash functions > Block and stream ciphers",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Symmetric cryptography and hash functions",
    "node_4": "Block and stream ciphers",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Symmetric cryptography and hash functions > Hash functions and message authentication codes",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Symmetric cryptography and hash functions",
    "node_4": "Hash functions and message authentication codes",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Cryptanalysis and other attacks",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Cryptanalysis and other attacks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Information-theoretic techniques",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Information-theoretic techniques",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Cryptography > Mathematical foundations of cryptography",
    "high_level_domain": "Security and privacy",
    "subdomain": "Cryptography",
    "node_3": "Mathematical foundations of cryptography",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Formal methods and theory of security > Trust frameworks",
    "high_level_domain": "Security and privacy",
    "subdomain": "Formal methods and theory of security",
    "node_3": "Trust frameworks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Formal methods and theory of security > Security requirements",
    "high_level_domain": "Security and privacy",
    "subdomain": "Formal methods and theory of security",
    "node_3": "Security requirements",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Formal methods and theory of security > Formal security models",
    "high_level_domain": "Security and privacy",
    "subdomain": "Formal methods and theory of security",
    "node_3": "Formal security models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Formal methods and theory of security > Logic and verification",
    "high_level_domain": "Security and privacy",
    "subdomain": "Formal methods and theory of security",
    "node_3": "Logic and verification",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Authentication",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Authentication",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Authentication > Biometrics",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Authentication",
    "node_4": "Biometrics",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Authentication > Graphical / visual passwords",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Authentication",
    "node_4": "Graphical / visual passwords",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Authentication > Multi-factor authentication",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Authentication",
    "node_4": "Multi-factor authentication",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Access control",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Access control",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Pseudonymity, anonymity and untraceability",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Pseudonymity, anonymity and untraceability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Privacy-preserving protocols",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Privacy-preserving protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Digital rights management",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Digital rights management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security services > Authorization",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security services",
    "node_3": "Authorization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Intrusion/anomaly detection and malware mitigation > Malware and its mitigation",
    "high_level_domain": "Security and privacy",
    "subdomain": "Intrusion/anomaly detection and malware mitigation",
    "node_3": "Malware and its mitigation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Intrusion/anomaly detection and malware mitigation > Intrusion detection systems",
    "high_level_domain": "Security and privacy",
    "subdomain": "Intrusion/anomaly detection and malware mitigation",
    "node_3": "Intrusion detection systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Intrusion/anomaly detection and malware mitigation > Artificial immune systems",
    "high_level_domain": "Security and privacy",
    "subdomain": "Intrusion/anomaly detection and malware mitigation",
    "node_3": "Artificial immune systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Intrusion/anomaly detection and malware mitigation > Social engineering attacks",
    "high_level_domain": "Security and privacy",
    "subdomain": "Intrusion/anomaly detection and malware mitigation",
    "node_3": "Social engineering attacks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Intrusion/anomaly detection and malware mitigation > Spoofing attacks",
    "high_level_domain": "Security and privacy",
    "subdomain": "Intrusion/anomaly detection and malware mitigation",
    "node_3": "Spoofing attacks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Intrusion/anomaly detection and malware mitigation > Phishing",
    "high_level_domain": "Security and privacy",
    "subdomain": "Intrusion/anomaly detection and malware mitigation",
    "node_3": "Phishing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Tamper-proof and tamper-resistant designs",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Tamper-proof and tamper-resistant designs",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Embedded systems security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Embedded systems security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Hardware security implementation",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Hardware security implementation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Hardware-based security protocols",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Hardware-based security protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Hardware attacks and countermeasures",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Hardware attacks and countermeasures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Hardware attacks and countermeasures > Malicious design modifications",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Hardware attacks and countermeasures",
    "node_4": "Malicious design modifications",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Hardware attacks and countermeasures > Side-channel analysis and countermeasures",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Hardware attacks and countermeasures",
    "node_4": "Side-channel analysis and countermeasures",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Security in hardware > Hardware attacks and countermeasures > Hardware reverse engineering",
    "high_level_domain": "Security and privacy",
    "subdomain": "Security in hardware",
    "node_3": "Hardware attacks and countermeasures",
    "node_4": "Hardware reverse engineering",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Operating systems security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Operating systems security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Mobile platform security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Mobile platform security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Trusted computing",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Trusted computing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Virtualization and security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Virtualization and security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Browser security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Browser security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Distributed systems security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Distributed systems security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Information flow control",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Information flow control",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Denial-of-service attacks",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Denial-of-service attacks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Firewalls",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Firewalls",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Vulnerability management",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Vulnerability management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Vulnerability management > Penetration testing",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Vulnerability management",
    "node_4": "Penetration testing",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > Vulnerability management > Vulnerability scanners",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "Vulnerability management",
    "node_4": "Vulnerability scanners",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Systems security > File system security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Systems security",
    "node_3": "File system security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Network security > Security protocols",
    "high_level_domain": "Security and privacy",
    "subdomain": "Network security",
    "node_3": "Security protocols",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Network security > Web protocol security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Network security",
    "node_3": "Web protocol security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Network security > Mobile and wireless security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Network security",
    "node_3": "Mobile and wireless security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Network security > Denial-of-service attacks",
    "high_level_domain": "Security and privacy",
    "subdomain": "Network security",
    "node_3": "Denial-of-service attacks",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Network security > Firewalls",
    "high_level_domain": "Security and privacy",
    "subdomain": "Network security",
    "node_3": "Firewalls",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Database and storage security > Data anonymization and sanitization",
    "high_level_domain": "Security and privacy",
    "subdomain": "Database and storage security",
    "node_3": "Data anonymization and sanitization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Database and storage security > Management and querying of encrypted data",
    "high_level_domain": "Security and privacy",
    "subdomain": "Database and storage security",
    "node_3": "Management and querying of encrypted data",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Database and storage security > Information accountability and usage control",
    "high_level_domain": "Security and privacy",
    "subdomain": "Database and storage security",
    "node_3": "Information accountability and usage control",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Database and storage security > Database activity monitoring",
    "high_level_domain": "Security and privacy",
    "subdomain": "Database and storage security",
    "node_3": "Database activity monitoring",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Software and application security > Software security engineering",
    "high_level_domain": "Security and privacy",
    "subdomain": "Software and application security",
    "node_3": "Software security engineering",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Software and application security > Web application security",
    "high_level_domain": "Security and privacy",
    "subdomain": "Software and application security",
    "node_3": "Web application security",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Software and application security > Social network security and privacy",
    "high_level_domain": "Security and privacy",
    "subdomain": "Software and application security",
    "node_3": "Social network security and privacy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Software and application security > Domain-specific security and privacy architectures",
    "high_level_domain": "Security and privacy",
    "subdomain": "Software and application security",
    "node_3": "Domain-specific security and privacy architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Software and application security > Software reverse engineering",
    "high_level_domain": "Security and privacy",
    "subdomain": "Software and application security",
    "node_3": "Software reverse engineering",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Human and societal aspects of security and privacy > Economics of security and privacy",
    "high_level_domain": "Security and privacy",
    "subdomain": "Human and societal aspects of security and privacy",
    "node_3": "Economics of security and privacy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Human and societal aspects of security and privacy > Social aspects of security and privacy",
    "high_level_domain": "Security and privacy",
    "subdomain": "Human and societal aspects of security and privacy",
    "node_3": "Social aspects of security and privacy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Human and societal aspects of security and privacy > Privacy protections",
    "high_level_domain": "Security and privacy",
    "subdomain": "Human and societal aspects of security and privacy",
    "node_3": "Privacy protections",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Security and privacy > Human and societal aspects of security and privacy > Usability in security and privacy",
    "high_level_domain": "Security and privacy",
    "subdomain": "Human and societal aspects of security and privacy",
    "node_3": "Usability in security and privacy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods > User models",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "User models",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods > User studies",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "User studies",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods > Usability testing",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "Usability testing",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods > Heuristic evaluations",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "Heuristic evaluations",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods > Walkthrough evaluations",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "Walkthrough evaluations",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods > Laboratory experiments",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "Laboratory experiments",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI design and evaluation methods > Field studies",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI design and evaluation methods",
    "node_4": "Field studies",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Hypertext / hypermedia",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Hypertext / hypermedia",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Mixed / augmented reality",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Mixed / augmented reality",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Command line interfaces",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Command line interfaces",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Graphical user interfaces",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Graphical user interfaces",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Virtual reality",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Virtual reality",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Web-based interaction",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Web-based interaction",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Natural language interfaces",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Natural language interfaces",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction paradigms > Collaborative interaction",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction paradigms",
    "node_4": "Collaborative interaction",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices > Graphics input devices",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "Graphics input devices",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices > Displays and imagers",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "Displays and imagers",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices > Sound-based input / output",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "Sound-based input / output",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices > Keyboards",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "Keyboards",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices > Pointing devices",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "Pointing devices",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices > Touch screens",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "Touch screens",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction devices > Haptic devices",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction devices",
    "node_4": "Haptic devices",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > HCI theory, concepts and models",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "HCI theory, concepts and models",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction techniques",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction techniques",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction techniques > Auditory feedback",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction techniques",
    "node_4": "Auditory feedback",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction techniques > Text input",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction techniques",
    "node_4": "Text input",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction techniques > Pointing",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction techniques",
    "node_4": "Pointing",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interaction techniques > Gestural input",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interaction techniques",
    "node_4": "Gestural input",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interactive systems and tools",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interactive systems and tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interactive systems and tools > User interface management systems",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interactive systems and tools",
    "node_4": "User interface management systems",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interactive systems and tools > User interface programming",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interactive systems and tools",
    "node_4": "User interface programming",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Interactive systems and tools > User interface toolkits",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Interactive systems and tools",
    "node_4": "User interface toolkits",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Human computer interaction (HCI) > Empirical studies in HCI",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Human computer interaction (HCI)",
    "node_3": "Empirical studies in HCI",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > Interaction design process and methods",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "Interaction design process and methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > User interface design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "User interface design",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > User interface design > User centered design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "User interface design",
    "node_4": "User centered design",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > User interface design > Activity centered design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "User interface design",
    "node_4": "Activity centered design",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > User interface design > Scenario-based design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "User interface design",
    "node_4": "Scenario-based design",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > User interface design > Participatory design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "User interface design",
    "node_4": "Participatory design",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > User interface design > Contextual design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "User interface design",
    "node_4": "Contextual design",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > User interface design > Interface design prototyping",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "User interface design",
    "node_4": "Interface design prototyping",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > Interaction design theory, concepts and paradigms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "Interaction design theory, concepts and paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > Empirical studies in interaction design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "Empirical studies in interaction design",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > Systems and tools for interaction design",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "Systems and tools for interaction design",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Interaction design > Systems and tools for interaction design > Wireframes",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Interaction design",
    "node_3": "Systems and tools for interaction design",
    "node_4": "Wireframes",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Social content sharing",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Social content sharing",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Collaborative content creation",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Collaborative content creation",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Collaborative filtering",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Collaborative filtering",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Social recommendation",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Social recommendation",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Social networks",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Social networks",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Social tagging",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Social tagging",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Computer supported cooperative work",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Computer supported cooperative work",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Social engineering (social sciences)",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Social engineering (social sciences)",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Social navigation",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Social navigation",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing theory, concepts and paradigms > Social media",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing theory, concepts and paradigms",
    "node_4": "Social media",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing design and evaluation methods",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing design and evaluation methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing design and evaluation methods > Social network analysis",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing design and evaluation methods",
    "node_4": "Social network analysis",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing design and evaluation methods > Ethnographic studies",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing design and evaluation methods",
    "node_4": "Ethnographic studies",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Blogs",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Blogs",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Wikis",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Wikis",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Reputation systems",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Reputation systems",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Open source software",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Open source software",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Social networking sites",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Social networking sites",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Social tagging systems",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Social tagging systems",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Synchronous editors",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Synchronous editors",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing systems and tools > Asynchronous editors",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing systems and tools",
    "node_4": "Asynchronous editors",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Empirical studies in collaborative and social computing",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Empirical studies in collaborative and social computing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Collaborative and social computing > Collaborative and social computing devices",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Collaborative and social computing",
    "node_3": "Collaborative and social computing devices",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing theory, concepts and paradigms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing theory, concepts and paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing theory, concepts and paradigms > Ubiquitous computing",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing theory, concepts and paradigms",
    "node_4": "Ubiquitous computing",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing theory, concepts and paradigms > Mobile computing",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing theory, concepts and paradigms",
    "node_4": "Mobile computing",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing theory, concepts and paradigms > Ambient intelligence",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing theory, concepts and paradigms",
    "node_4": "Ambient intelligence",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Ubiquitous and mobile devices",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Ubiquitous and mobile devices",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Smartphones",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Smartphones",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Interactive whiteboards",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Interactive whiteboards",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Mobile phones",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Mobile phones",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Mobile devices",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Mobile devices",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Portable media players",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Portable media players",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Personal digital assistants",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Personal digital assistants",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Handheld game consoles",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Handheld game consoles",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > E-book readers",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "E-book readers",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing systems and tools > Tablet computers",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing systems and tools",
    "node_4": "Tablet computers",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Ubiquitous and mobile computing design and evaluation methods",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Ubiquitous and mobile computing design and evaluation methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Ubiquitous and mobile computing > Empirical studies in ubiquitous and mobile computing",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Ubiquitous and mobile computing",
    "node_3": "Empirical studies in ubiquitous and mobile computing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization techniques",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization techniques",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization techniques > Treemaps",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization techniques",
    "node_4": "Treemaps",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization techniques > Hyperbolic trees",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization techniques",
    "node_4": "Hyperbolic trees",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization techniques > Heat maps",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization techniques",
    "node_4": "Heat maps",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization techniques > Graph drawings",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization techniques",
    "node_4": "Graph drawings",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization techniques > Dendrograms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization techniques",
    "node_4": "Dendrograms",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization techniques > Cladograms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization techniques",
    "node_4": "Cladograms",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization application domains",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization application domains",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization application domains > Scientific visualization",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization application domains",
    "node_4": "Scientific visualization",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization application domains > Visual analytics",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization application domains",
    "node_4": "Visual analytics",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization application domains > Geographic visualization",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization application domains",
    "node_4": "Geographic visualization",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization application domains > Information visualization",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization application domains",
    "node_4": "Information visualization",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization systems and tools",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization systems and tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization systems and tools > Visualization toolkits",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization systems and tools",
    "node_4": "Visualization toolkits",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization theory, concepts and paradigms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization theory, concepts and paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Empirical studies in visualization",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Empirical studies in visualization",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Visualization > Visualization design and evaluation methods",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Visualization",
    "node_3": "Visualization design and evaluation methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Accessibility > Accessibility theory, concepts and paradigms",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Accessibility",
    "node_3": "Accessibility theory, concepts and paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Accessibility > Empirical studies in accessibility",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Accessibility",
    "node_3": "Empirical studies in accessibility",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Accessibility > Accessibility design and evaluation methods",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Accessibility",
    "node_3": "Accessibility design and evaluation methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Accessibility > Accessibility technologies",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Accessibility",
    "node_3": "Accessibility technologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Human-centered computing > Accessibility > Accessibility systems and tools",
    "high_level_domain": "Human-centered computing",
    "subdomain": "Accessibility",
    "node_3": "Accessibility systems and tools",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Combinatorial algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Combinatorial algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Algebraic algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Algebraic algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Nonalgebraic algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Nonalgebraic algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Symbolic calculus algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Symbolic calculus algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Exact arithmetic algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Exact arithmetic algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Hybrid symbolic-numeric methods",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Hybrid symbolic-numeric methods",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Discrete calculus algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Discrete calculus algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Number theory algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Number theory algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Equation and inequality solving algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Equation and inequality solving algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Linear algebra algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Linear algebra algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Theorem proving algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Theorem proving algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Boolean algebra algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Boolean algebra algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Symbolic and algebraic algorithms > Optimization algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Symbolic and algebraic algorithms",
    "node_4": "Optimization algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Computer algebra systems",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Computer algebra systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Computer algebra systems > Special-purpose algebraic systems",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Computer algebra systems",
    "node_4": "Special-purpose algebraic systems",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Representation of mathematical objects",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Representation of mathematical objects",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Representation of mathematical objects > Representation of exact numbers",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Representation of mathematical objects",
    "node_4": "Representation of exact numbers",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Representation of mathematical objects > Representation of mathematical functions",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Representation of mathematical objects",
    "node_4": "Representation of mathematical functions",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Representation of mathematical objects > Representation of Boolean functions",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Representation of mathematical objects",
    "node_4": "Representation of Boolean functions",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Symbolic and algebraic manipulation > Representation of mathematical objects > Representation of polynomials",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Symbolic and algebraic manipulation",
    "node_3": "Representation of mathematical objects",
    "node_4": "Representation of polynomials",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Parallel computing methodologies > Parallel algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Parallel computing methodologies",
    "node_3": "Parallel algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Parallel computing methodologies > Parallel algorithms > MapReduce algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Parallel computing methodologies",
    "node_3": "Parallel algorithms",
    "node_4": "MapReduce algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Parallel computing methodologies > Parallel algorithms > Self-organization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Parallel computing methodologies",
    "node_3": "Parallel algorithms",
    "node_4": "Self-organization",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Parallel computing methodologies > Parallel algorithms > Shared memory algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Parallel computing methodologies",
    "node_3": "Parallel algorithms",
    "node_4": "Shared memory algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Parallel computing methodologies > Parallel algorithms > Vector / streaming algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Parallel computing methodologies",
    "node_3": "Parallel algorithms",
    "node_4": "Vector / streaming algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Parallel computing methodologies > Parallel algorithms > Massively parallel algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Parallel computing methodologies",
    "node_3": "Parallel algorithms",
    "node_4": "Massively parallel algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Parallel computing methodologies > Parallel programming languages",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Parallel computing methodologies",
    "node_3": "Parallel programming languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Information extraction",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Information extraction",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Machine translation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Machine translation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Discourse, dialogue and pragmatics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Discourse, dialogue and pragmatics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Natural language generation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Natural language generation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Speech recognition",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Speech recognition",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Lexical semantics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Lexical semantics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Phonology / morphology",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Phonology / morphology",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Natural language processing > Language resources",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Natural language processing",
    "node_4": "Language resources",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Description logics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Description logics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Semantic networks",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Semantic networks",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Nonmonotonic, default reasoning and belief revision",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Nonmonotonic, default reasoning and belief revision",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Probabilistic reasoning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Probabilistic reasoning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Vagueness and fuzzy logic",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Vagueness and fuzzy logic",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Causal reasoning and diagnostics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Causal reasoning and diagnostics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Temporal reasoning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Temporal reasoning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Cognitive robotics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Cognitive robotics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Ontology engineering",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Ontology engineering",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Logic programming and answer set programming",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Logic programming and answer set programming",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Spatial and physical reasoning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Spatial and physical reasoning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Knowledge representation and reasoning > Reasoning about belief and knowledge",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Knowledge representation and reasoning",
    "node_4": "Reasoning about belief and knowledge",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Planning and scheduling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Planning and scheduling",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Planning and scheduling > Planning for deterministic actions",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Planning and scheduling",
    "node_4": "Planning for deterministic actions",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Planning and scheduling > Planning under uncertainty",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Planning and scheduling",
    "node_4": "Planning under uncertainty",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Planning and scheduling > Multi-agent planning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Planning and scheduling",
    "node_4": "Multi-agent planning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Planning and scheduling > Planning with abstraction and generalization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Planning and scheduling",
    "node_4": "Planning with abstraction and generalization",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Planning and scheduling > Robotic planning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Planning and scheduling",
    "node_4": "Robotic planning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Planning and scheduling > Evolutionary robotics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Planning and scheduling",
    "node_4": "Evolutionary robotics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies > Heuristic function construction",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "Heuristic function construction",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies > Discrete space search",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "Discrete space search",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies > Continuous space search",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "Continuous space search",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies > Randomized search",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "Randomized search",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies > Game tree search",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "Game tree search",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies > Abstraction and micro-operators",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "Abstraction and micro-operators",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Search methodologies > Search with partial observations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Search methodologies",
    "node_4": "Search with partial observations",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Control methods",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Control methods",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Control methods > Robotic planning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Control methods",
    "node_4": "Robotic planning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Control methods > Evolutionary robotics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Control methods",
    "node_4": "Evolutionary robotics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Control methods > Computational control theory",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Control methods",
    "node_4": "Computational control theory",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Control methods > Motion path planning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Control methods",
    "node_4": "Motion path planning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Philosophical/theoretical foundations of artificial intelligence",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Philosophical/theoretical foundations of artificial intelligence",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Philosophical/theoretical foundations of artificial intelligence > Cognitive science",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Philosophical/theoretical foundations of artificial intelligence",
    "node_4": "Cognitive science",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Philosophical/theoretical foundations of artificial intelligence > Theory of mind",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Philosophical/theoretical foundations of artificial intelligence",
    "node_4": "Theory of mind",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Distributed artificial intelligence",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Distributed artificial intelligence",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Distributed artificial intelligence > Multi-agent systems",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Distributed artificial intelligence",
    "node_4": "Multi-agent systems",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Distributed artificial intelligence > Intelligent agents",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Distributed artificial intelligence",
    "node_4": "Intelligent agents",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Distributed artificial intelligence > Mobile agents",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Distributed artificial intelligence",
    "node_4": "Mobile agents",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Distributed artificial intelligence > Cooperation and coordination",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Distributed artificial intelligence",
    "node_4": "Cooperation and coordination",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision tasks",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision tasks > Biometrics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Biometrics"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision tasks > Scene understanding",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Scene understanding"
  },
  {
    "path": "Computing methodologies > Artificial intelligence> > Computer vision > Computer vision tasks > Activity recognition and understanding",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Activity recognition and understanding"
  },
  {
    "path": "Computing methodologies > Artificial intelligence> Computer vision > Computer vision tasks > Video summarization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Video summarization"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision tasks > Visual content-based indexing and retrieval",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Visual content-based indexing and retrieval"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision tasks > Visual inspection",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Visual inspection"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision tasks > Vision for robotics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Vision for robotics"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision tasks > Scene anomaly detection",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision tasks",
    "node_5": "Scene anomaly detection"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition > Camera calibration",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": "Camera calibration"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition > Epipolar geometry",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": "Epipolar geometry"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition > Computational photography",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": "Computational photography"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition > Hyperspectral imaging",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": "Hyperspectral imaging"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition > Motion capture",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": "Motion capture"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition > 3D imaging",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": "3D imaging"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Image and video acquisition > Active vision",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Image and video acquisition",
    "node_5": "Active vision"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision representations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision representations",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision representations > Image representations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision representations",
    "node_5": "Image representation"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision representations > Shape representations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision representations",
    "node_5": "Shape representation"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision representations > Appearance and texture representations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision representations",
    "node_5": "Appearance and texture representations"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision representations > Hierarchical representations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision representations",
    "node_5": "Hierarchical representations"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Interest point and salient region detections",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Interest point and salient region detections"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Image segmentation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Image segmentation"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Video segmentation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Video segmentation"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Shape inference",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Shape inference"
  },
  {
    "path": "Computing methodologies > Artificial intelligence >Computer vision > Computer vision problems > Object detection",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Object detection"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Object recognition",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Object recognition"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Object identification",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Object identification"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Tracking",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Tracking"
  },
  {
    "path": "Computing methodologies > Artificial intelligence >Computer vision > Computer vision problems > Reconstruction",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Reconstruction"
  },
  {
    "path": "Computing methodologies > Artificial intelligence > Computer vision > Computer vision problems > Matching",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Artificial intelligence",
    "node_3": "Computer vision",
    "node_4": "Computer vision problems",
    "node_5": "Matching"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Supervised learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Supervised learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Supervised learning > Ranking",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Supervised learning",
    "node_5": "Ranking"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Supervised learning > Learning to rank",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Supervised learning",
    "node_5": "Learning to rank"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Supervised learning > Supervised learning by classification",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Supervised learning",
    "node_5": "Supervised learning by classification"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Supervised learning > Supervised learning by regression",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Supervised learning",
    "node_5": "Supervised learning by regression"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Supervised learning > Structured outputs",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Supervised learning",
    "node_5": "Structured outputs"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Supervised learning > Cost-sensitive learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Supervised learning",
    "node_5": "Cost-sensitive learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning > Cluster analysis",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": "Cluster analysis"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning > Anomaly detection",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": "Anomaly detection"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning > Mixture modeling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": "Mixture modeling"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning > Topic modeling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": "Topic modeling"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning > Source separation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": "Source separation"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning > Motif discovery",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": "Motif discovery"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Unsupervised learning > Dimensionality reduction and manifold learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Unsupervised learning",
    "node_5": "Dimensionality reduction and manifold learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Reinforcement learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Reinforcement learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Reinforcement learning > Sequential decision making",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Reinforcement learning",
    "node_5": "Sequential decision making"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Reinforcement learning > Inverse reinforcement learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Reinforcement learning",
    "node_5": "Inverse reinforcement learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Reinforcement learning > Apprenticeship learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Reinforcement learning",
    "node_5": "Apprenticeship learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Reinforcement learning > Multi-agent reinforcement learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Reinforcement learning",
    "node_5": "Multi-agent reinforcement learnin"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Reinforcement learning > Adversarial learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Reinforcement learning",
    "node_5": "Adversarial learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Multi-task learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Multi-task learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Multi-task learning > Transfer learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Multi-task learning",
    "node_5": "Transfer learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Multi-task learning > Lifelong machine learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Multi-task learning",
    "node_5": "Lifelong machine learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning paradigms > Multi-task learning > Learning under covariate shift",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning paradigms",
    "node_4": "Multi-task learning",
    "node_5": "Learning under covariate shift"
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings > Batch learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "Batch learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings > Online learning settings",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "Online learning settings",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings > Learning from demonstrations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "Learning from demonstrations",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings > Learning from critiques",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "Learning from critiques",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings > Learning from implicit feedback",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "Learning from implicit feedback",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings > Active learning settings",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "Active learning settings",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Learning settings > Semi-supervised learning settings",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Learning settings",
    "node_4": "Semi-supervised learning settings",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Classification and regression trees",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Classification and regression trees",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Kernel methods",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Kernel methods",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Kernel methods > Support vector machines",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Kernel methods",
    "node_5": "Support vector machines"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Kernel methods > Gaussian processes",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Kernel methods",
    "node_5": "Gaussian processes"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Neural networks",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Neural networks",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Logical and relational learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Logical and relational learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Logical and relational learning > Inductive logic learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Logical and relational learning",
    "node_5": "Inductive logic learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Logical and relational learning > Statistical relational learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Logical and relational learning",
    "node_5": "Statistical relational learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning in probabilistic graphical models > Maximum likelihood modeling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning in probabilistic graphical models",
    "node_5": "Maximum likelihood modeling"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning in probabilistic graphical models >",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning in probabilistic graphical models",
    "node_5": "Maximum likelihood modeling"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Maximum likelihood modeling  > ",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning in probabilistic graphical models",
    "node_5": "Maximum entropy modeling"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning in probabilistic graphical models > Maximum entropy modeling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning in probabilistic graphical models",
    "node_5": "Maximum a posteriori modeling"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning in probabilistic graphical models> Maximum a posteriori modeling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Probabilistic and relational learning",
    "node_4": "Maximum a posteriori modeling",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning in probabilistic graphical models > Mixture models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning in probabilistic graphical models",
    "node_5": "Mixture models"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning in probabilistic graphical models > Latent variable models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning in probabilistic graphical models",
    "node_5": "Latent variable mode"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning in probabilistic graphical models > Bayesian network models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning in probabilistic graphical models",
    "node_5": "Bayesian network model"
  },
  {
    "node_4": "Learning linear models",
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning linear models > Bayesian network models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning linear models",
    "node_5": "Bayesian network model"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Factorization methods",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Factorization methods",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Factorization methods > Non-negative matrix factorization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Factorization methods",
    "node_5": "Non-negative matrix factorization"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Factorization methods > Factor analysis",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Factorization methods",
    "node_5": "Factor analysis"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Factorization methods > Principal component analysis",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Factorization methods",
    "node_5": "Principal component analysis"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Factorization methods > Canonical correlation analysis",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Factorization methods",
    "node_5": "Canonical correlation analysis"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Factorization methods > Latent Dirichlet allocation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Factorization methods",
    "node_5": "Latent Dirichlet allocation"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Rule learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Rule learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Instance-based learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Instance-based learning",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Markov decision processes",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Markov decision processes",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Partially-observable Markov decision processes",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Partially-observable Markov decision processes",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Stochastic games",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Stochastic games",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning latent representations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning latent representations",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Learning latent representations > Deep belief networks",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Learning latent representations",
    "node_5": "Deep belief networks"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Bio-inspired approaches",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Bio-inspired approaches",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Bio-inspired approaches > Artificial life",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Bio-inspired approaches",
    "node_5": "Artificial life"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Bio-inspired approaches > Evolvable hardware",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Bio-inspired approaches",
    "node_5": "Evolvable hardware"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Bio-inspired approaches > Genetic algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Bio-inspired approaches",
    "node_5": "Genetic algorithms"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Bio-inspired approaches > Genetic programming",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Bio-inspired approaches",
    "node_5": "Genetic programming"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Bio-inspired approaches > Evolutionary robotics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Bio-inspired approaches",
    "node_5": "Evolutionary robotics"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning approaches > Bio-inspired approaches > Generative and developmental approaches",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning approaches",
    "node_4": "Bio-inspired approaches",
    "node_5": "Generative and developmental approaches"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Dynamic programming for Markov decision processes",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Dynamic programming for Markov decision processes",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Dynamic programming for Markov decision processes > Value iteration",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Dynamic programming for Markov decision processes",
    "node_5": "Value iteration"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Dynamic programming for Markov decision processes > Q-learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Dynamic programming for Markov decision processes",
    "node_5": "Q-learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Dynamic programming for Markov decision processes > Policy iteration",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Dynamic programming for Markov decision processes",
    "node_5": "Policy iteration"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Dynamic programming for Markov decision processes > Temporal difference learning",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Dynamic programming for Markov decision processes",
    "node_5": "Temporal difference learning"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Dynamic programming for Markov decision processes > Approximate dynamic programming methods",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Dynamic programming for Markov decision processes",
    "node_5": "Approximate dynamic programming methods"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Ensemble methods",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Ensemble methods",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Ensemble methods > Boosting",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Ensemble methods",
    "node_5": "Boosting"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Ensemble methods > Bagging",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Ensemble methods",
    "node_5": "Bagging"
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Spectral methods",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Spectral methods",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Feature selection",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Feature selection",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Machine learning algorithms > Regularization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Machine learning algorithms",
    "node_4": "Regularization",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Machine learning > Cross-validation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Machine learning",
    "node_3": "Cross-validation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Model development and analysis",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Model development and analysis",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Model development and analysis > Modeling methodologies",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Model development and analysis",
    "node_4": "Modeling methodologies",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Model development and analysis > Model verification and validation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Model development and analysis",
    "node_4": "Model verification and validation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Model development and analysis > Uncertainty quantification",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Model development and analysis",
    "node_4": "Uncertainty quantification",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation theory",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation theory",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation theory > Systems theory",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation theory",
    "node_4": "Systems theory",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation theory > Network science",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation theory",
    "node_4": "Network science",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Quantum mechanic simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Quantum mechanic simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Molecular simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Molecular simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Rare-event simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Rare-event simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Discrete-event simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Discrete-event simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Agent / discrete models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Agent / discrete models",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Distributed simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Distributed simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Continuous simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Continuous simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Continuous models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Continuous models",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Real-time simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Real-time simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Interactive simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Interactive simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Multiscale systems",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Multiscale systems",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Massively parallel and high-performance simulations",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Massively parallel and high-performance simulations",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Data assimilation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Data assimilation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Scientific visualization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Scientific visualization",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Visual analytics",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Visual analytics",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Simulation by animation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Simulation by animation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation types and techniques > Artificial life",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation types and techniques",
    "node_4": "Artificial life",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation support systems",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation support systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation support systems > Simulation environments",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation support systems",
    "node_4": "Simulation environments",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation support systems > Simulation languages",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation support systems",
    "node_4": "Simulation languages",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation support systems > Simulation tools",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation support systems",
    "node_4": "Simulation tools",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Modeling and simulation > Simulation evaluation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Modeling and simulation",
    "node_3": "Simulation evaluation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Animation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Animation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Animation > Motion capture",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Animation",
    "node_4": "Motion capture",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Animation > Procedural animation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Animation",
    "node_4": "Procedural animation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Animation > Physical simulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Animation",
    "node_4": "Physical simulation",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Animation > Motion processing",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Animation",
    "node_4": "Motion processing",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Animation > Collision detection",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Animation",
    "node_4": "Collision detection",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics >  Rendering",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Rendering",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Rendering > Rasterization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Rendering",
    "node_4": "Rasterization",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Rendering > Ray tracing",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Rendering",
    "node_4": "Ray tracing",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Rendering > Non-photorealistic rendering",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Rendering",
    "node_4": "Non-photorealistic rendering",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Rendering > Reflectance modeling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Rendering",
    "node_4": "Reflectance modeling",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Rendering > Visibility",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Rendering",
    "node_4": "Visibility",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Image manipulation",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Image manipulation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics >  Image manipulation  > Image processing",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Image manipulatioon",
    "node_4": "Image processing",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Image manipulation > Computational photography",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Image manipulation",
    "node_4": "Computational photography",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Image manipulation > Texturing",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Image manipulation",
    "node_4": "Texturing",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Image processing and manipulation > Image-based rendering",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Image processing and manipulation",
    "node_4": "Image-based rendering",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Image manipulation > Antialiasing",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Image manipulation",
    "node_4": "Antialiasing",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces > Graphics processors",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "Graphics processors",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces > Graphics input devices",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "Graphics input devices",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces > Mixed / augmented reality",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "Mixed / augmented reality",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces > Perception",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "Perception",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces > Graphics file formats",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "Graphics file formats",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces > Virtual reality",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "Virtual reality",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Graphics systems and interfaces > Image compression",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Graphics systems and interfaces",
    "node_4": "Image compression",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Shape modeling",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Shape modeling",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Shape modeling > Mesh models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Shape modeling",
    "node_4": "Mesh models",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Shape modeling > Mesh geometry models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Shape modeling",
    "node_4": "Mesh geometry models",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Shape modeling > Parametric curve and surface models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Shape modeling",
    "node_4": "Parametric curve and surface models",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Shape modeling > Point-based models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Shape modeling",
    "node_4": "Point-based models",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Shape modeling > Volumetric models",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Shape modeling",
    "node_4": "Volumetric models",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Computer graphics > Shape modeling > Shape analysis",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Computer graphics",
    "node_3": "Shape modeling",
    "node_4": "Shape analysis",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Distributed computing methodologies > Distributed algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Distributed computing methodologies",
    "node_3": "",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Distributed computing methodologies > Distributed algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Distributed computing methodologies",
    "node_3": "Distributed algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Distributed computing methodologies > Distributed algorithms > MapReduce algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Distributed computing methodologies",
    "node_3": "Distributed algorithms",
    "node_4": "MapReduce algorithms",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Distributed computing methodologies > Distributed algorithms >  Self-organization",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Distributed computing methodologies",
    "node_3": "Distributed algorithms",
    "node_4": "Self-organization",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Distributed computing methodologies > Distributed programming languages",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Distributed computing methodologies",
    "node_3": "Distributed programming languages",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies >  Concurrent computing methodologies",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Distributed computing methodologies",
    "node_3": "Concurrent computing methodologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies > Distributed computing methodologies > Concurrent computing methodologies > Concurrent programming languages",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Distributed computing methodologies",
    "node_3": "Concurrent computing methodologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Computing methodologies >  Concurrent computing methodologies > Concurrent algorithms",
    "high_level_domain": "Computing methodologies",
    "subdomain": "Concurrent computing methodologies",
    "node_3": "Concurrent algorithms",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > Digital cash",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "Digital cash",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > E-commerce infrastructure",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "E-commerce infrastructure",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > Electronic data interchange",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "Electronic data interchange",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > Electronic funds transfer",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "Electronic funds transfer",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > Online shopping",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "Online shopping",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > Online banking",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "Online banking",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > Secure online transactions",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "Secure online transactions",
    "node_5": ""
  },
  {
    "path": "Applied computing > Electronic commerce > E-commerce technologies > Online auctions",
    "high_level_domain": "Applied computing",
    "subdomain": "Electronic commerce",
    "node_3": "E-commerce technologies",
    "node_4": "Online auctions",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise information systems",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise information systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise information systems > Intranets",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise information systems",
    "node_4": "Intranets",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise information systems > Extranets",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise information systems",
    "node_4": "Extranets",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise information systems > Enterprise resource planning",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise information systems",
    "node_4": "Enterprise resource planning",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise information systems > Enterprise applications",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise information systems",
    "node_4": "Enterprise applications",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise information systems > Data centers",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise information systems",
    "node_4": "Data centers",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Business process management",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Business process management",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Business process management > Business process modeling",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Business process management",
    "node_4": "Business process modeling",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Business process management > Business process management systems",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Business process management",
    "node_4": "Business process management systems",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Business process management > Business process monitoring",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Business process management",
    "node_4": "Business process monitoring",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Business process management > Cross-organizational business processes",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Business process management",
    "node_4": "Cross-organizational business processes",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Business intelligence",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Business intelligence",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Enterprise architecture management",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Enterprise architecture management",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Enterprise architecture frameworks",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Enterprise architecture frameworks",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Enterprise architecture modeling",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Enterprise architecture modeling",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Service-oriented architectures",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Service-oriented architectures",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Event-driven architectures",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Event-driven architectures",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Business rules",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Business rules",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Enterprise modeling",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Enterprise modeling",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Enterprise ontologies, taxonomies and vocabularies",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Enterprise ontologies, taxonomies and vocabularies",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Enterprise data management",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Enterprise data management",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Reference models",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Reference models",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > Business-IT alignment",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "Business-IT alignment",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > IT architectures",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "IT architectures",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise architectures > IT governance",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise architectures",
    "node_4": "IT governance",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise computing infrastructures",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise computing infrastructures",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise interoperability",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise interoperability",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise interoperability > Enterprise application integration",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise interoperability",
    "node_4": "Enterprise application integration",
    "node_5": ""
  },
  {
    "path": "Applied computing > Enterprise computing > Enterprise interoperability > Information integration and interoperability",
    "high_level_domain": "Applied computing",
    "subdomain": "Enterprise computing",
    "node_3": "Enterprise interoperability",
    "node_4": "Information integration and interoperability",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Aerospace",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Aerospace",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Avionics",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Avionics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Archaeology",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Archaeology",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Astronomy",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Astronomy",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Chemistry",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Chemistry",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Earth and atmospheric sciences",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Earth and atmospheric sciences",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Environmental sciences",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Environmental sciences",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Engineering",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Engineering",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Computer-aided design",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Computer-aided design",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Physics",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Physics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Mathematics and statistics",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Mathematics and statistics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Electronics",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Electronics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Telecommunications",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Telecommunications",
    "node_5": ""
  },
  {
    "path": "Applied computing > Physical sciences and engineering > Scientific domains > Internet telephony",
    "high_level_domain": "Applied computing",
    "subdomain": "Physical sciences and engineering",
    "node_3": "Scientific domains",
    "node_4": "Internet telephony",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Molecular sequence analysis",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Molecular sequence analysis",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Recognition of genes and regulatory elements",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Recognition of genes and regulatory elements",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Molecular evolution",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Molecular evolution",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Computational transcriptomics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Computational transcriptomics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Biological networks",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Biological networks",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Sequencing and genotyping technologies",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Sequencing and genotyping technologies",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Imaging",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Imaging",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Computational proteomics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Computational proteomics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Molecular structural biology",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Molecular structural biology",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Computational genomics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Computational genomics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Genomics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Genomics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Computational biology > Systems biology",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Computational biology",
    "node_4": "Systems biology",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Consumer health",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Consumer health",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Health care information systems",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Health care information systems",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Health informatics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Health informatics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Bioinformatics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Bioinformatics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Metabolomics / metabonomics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Metabolomics / metabonomics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Genetics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Genetics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Population genetics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Population genetics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Proteomics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Proteomics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Life and medical sciences > Health informatics > Transcriptomics",
    "high_level_domain": "Applied computing",
    "subdomain": "Life and medical sciences",
    "node_3": "Health informatics",
    "node_4": "Transcriptomics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Social sciences",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Social sciences",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Social sciences > Anthropology",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Social sciences",
    "node_4": "Anthropology",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Social sciences > Ethnography",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Social sciences",
    "node_4": "Ethnography",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Social sciences > Law",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Social sciences",
    "node_4": "Law",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Social sciences > Psychology",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Social sciences",
    "node_4": "Psychology",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Social sciences > Economics",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Social sciences",
    "node_4": "Economics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Social sciences > Sociology",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Social sciences",
    "node_4": "Sociology",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Computer forensics",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Computer forensics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Computer forensics > Surveillance mechanisms",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Computer forensics",
    "node_4": "Surveillance mechanisms",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Computer forensics > Investigation techniques",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Computer forensics",
    "node_4": "Investigation techniques",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Computer forensics > Evidence collection, storage and analysis",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Computer forensics",
    "node_4": "Evidence collection, storage and analysis",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Computer forensics > Network forensics",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Computer forensics",
    "node_4": "Network forensics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Computer forensics > System forensics",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Computer forensics",
    "node_4": "System forensics",
    "node_5": ""
  },
  {
    "path": "Applied computing > Law, social and behavioral sciences > Computer forensics > Data recovery",
    "high_level_domain": "Applied computing",
    "subdomain": "Law, social and behavioral sciences",
    "node_3": "Computer forensics",
    "node_4": "Data recovery",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains > Fine arts",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "Fine arts",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains > Performing arts",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "Performing arts",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains > Architecture (buildings)",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "Architecture (buildings)",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains > Computer-aided design",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "Computer-aided design",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains > Language translation",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "Language translation",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains > Media arts",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "Media arts",
    "node_5": ""
  },
  {
    "path": "Applied computing > Arts and humanities > Arts domains > Sound and music computing",
    "high_level_domain": "Applied computing",
    "subdomain": "Arts and humanities",
    "node_3": "Arts domains",
    "node_4": "Sound and music computing",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Digital libraries and archives",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Digital libraries and archives",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Publishing",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Publishing",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Military",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Military",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Cyberwarfare",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Cyberwarfare",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Cartography",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Cartography",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Agriculture",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Agriculture",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Computing in government",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Computing in government",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > Voting / election technologies",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "Voting / election technologies",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > General applications > E-government",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "General applications",
    "node_4": "E-government",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Personal computing applications",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Personal computing applications",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Personal computing applications > Personal computers and PC applications",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Personal computing applications",
    "node_4": "Personal computers and PC applications",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Personal computing applications > Word processors",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Personal computing applications",
    "node_4": "Word processors",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Personal computing applications > Spreadsheets",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Personal computing applications",
    "node_4": "Spreadsheets",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Personal computing applications > Computer games",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Personal computing applications",
    "node_4": "Computer games",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Personal computing applications > Microcomputers",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Personal computing applications",
    "node_4": "Microcomputers",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Operations research",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Operations research",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Consumer products",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Consumer products",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Industry and manufacturing",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Industry and manufacturing",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Supply chain management",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Supply chain management",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Command and control",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Command and control",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Computer-aided manufacturing",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Computer-aided manufacturing",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Decision analysis",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Decision analysis",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Multi-criterion optimization and decision-making",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Multi-criterion optimization and decision-making",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Transportation",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Transportation",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Forecasting",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Forecasting",
    "node_5": ""
  },
  {
    "path": "Applied computing > Computers in other domains > Industry and operations > Marketing",
    "high_level_domain": "Applied computing",
    "subdomain": "Computers in other domains",
    "node_3": "Industry and operations",
    "node_4": "Marketing",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > Digital libraries and archives",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "Digital libraries and archives",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > Computer-assisted instruction",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "Computer-assisted instruction",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > Interactive learning environments",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "Interactive learning environments",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > Collaborative learning",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "Collaborative learning",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > Learning management systems",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "Learning management systems",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > Distance learning",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "Distance learning",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > E-learning",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "E-learning",
    "node_5": ""
  },
  {
    "path": "Applied computing > Education > Educational technologies > Computer-managed instruction",
    "high_level_domain": "Applied computing",
    "subdomain": "Education",
    "node_3": "Educational technologies",
    "node_4": "Computer-managed instruction",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Document searching",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Document searching",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Document management",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Document management",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Text editing",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Text editing",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Version control",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Version control",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Document metadata",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Document metadata",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Document capture",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Document capture",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Document analysis",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Document analysis",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document processing > Document scanning",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document processing",
    "node_4": "Document scanning",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Recognition and interpretation",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Recognition and interpretation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Recognition and interpretation > Graphics recognition and interpretation",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Recognition and interpretation",
    "node_4": "Graphics recognition and interpretation",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Recognition and interpretation > Optical character recognition",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Recognition and interpretation",
    "node_4": "Optical character recognition",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Recognition and interpretation > Online handwriting recognition",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Recognition and interpretation",
    "node_4": "Online handwriting recognition",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Markup languages",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Markup languages",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Extensible Markup Language (XML)",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Extensible Markup Language (XML)",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Hypertext languages",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Hypertext languages",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Annotation",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Annotation",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Format and notation",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Format and notation",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Multi / mixed media creation",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Multi / mixed media creation",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Image composition",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Image composition",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Hypertext / hypermedia creation",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Hypertext / hypermedia creation",
    "node_5": ""
  },
  {
    "path": "Applied computing > Document management and text processing > Document preparation > Document scripting languages",
    "high_level_domain": "Applied computing",
    "subdomain": "Document management and text processing",
    "node_3": "Document preparation",
    "node_4": "Document scripting languages",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Computing industry",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Computing industry",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Computing industry > Industry statistics",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Computing industry",
    "node_4": "Industry statistics",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Computing industry > Computer manufacturing",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Computing industry",
    "node_4": "Computer manufacturing",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Computing industry > Sustainability",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Computing industry",
    "node_4": "Sustainability",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Project and people management",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Project and people management",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Project management techniques",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Project management techniques",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Project staffing",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Project staffing",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Systems planning",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Systems planning",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Systems analysis and design",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Systems analysis and design",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Systems development",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Systems development",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Computer and information systems training",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Computer and information systems training",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Implementation management",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Implementation management",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Hardware selection",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Hardware selection",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Computing equipment management",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Computing equipment management",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Pricing and resource allocation",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Pricing and resource allocation",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Software management",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Software management",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Software maintenance",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Software maintenance",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Software selection and adaptation",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Software selection and adaptation",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > System management",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "System management",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Centralization / decentralization",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Centralization / decentralization",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Technology audits",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Technology audits",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Quality assurance",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Quality assurance",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Network operations",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Network operations",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > File systems management",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "File systems management",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > Management of computing and information systems > Information system economics",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "Management of computing and information systems",
    "node_4": "Information system economics",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > History of computing",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "History of computing",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > History of computing > Historical people",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "History of computing",
    "node_4": "Historical people",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > History of computing > History of hardware",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "History of computing",
    "node_4": "History of hardware",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > History of computing > History of software",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "History of computing",
    "node_4": "History of software",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > History of computing > History of programming languages",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "History of computing",
    "node_4": "History of programming languages",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Professional topics > History of computing > History of computing theory",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Professional topics",
    "node_3": "History of computing",
    "node_4": "History of computing theory",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Educational foundations",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Educational foundations",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Educational foundations > Computational thinking",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Educational foundations",
    "node_4": "Computational thinking",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Educational foundations > Accreditation",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Educational foundations",
    "node_4": "Accreditation",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Educational foundations > Model curricula",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Educational foundations",
    "node_4": "Model curricula",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > Information systems education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "Information systems education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > Computer science education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "Computer science education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > CS1",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "CS1",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > Computer engineering education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "Computer engineering education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > Information technology education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "Information technology education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > Information science education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "Information science education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > Computational science and engineering education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "Computational science and engineering education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Computing education programs > Software engineering education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Computing education programs",
    "node_4": "Software engineering education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Learning and assessment",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Learning and assessment",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Learning and assessment > Informal education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Learning and assessment",
    "node_4": "Informal education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Learning and assessment > Computing literacy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Learning and assessment",
    "node_4": "Computing literacy",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Learning and assessment > Student assessment",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Learning and assessment",
    "node_4": "Student assessment",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Learning and assessment > K-12 education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Learning and assessment",
    "node_4": "K-12 education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing education > Learning and assessment > Adult education",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing education",
    "node_3": "Learning and assessment",
    "node_4": "Adult education",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects > Employment issues",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "Employment issues",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects > Automation",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "Automation",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects > Computer supported cooperative work",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "Computer supported cooperative work",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects > Economic impact",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "Economic impact",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects > Offshoring",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "Offshoring",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects > Reengineering",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "Reengineering",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing and business > Economic and organizational aspects > Socio-technical systems",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing and business",
    "node_3": "Economic and organizational aspects",
    "node_4": "Socio-technical systems",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues > Codes of ethics",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "Codes of ethics",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues > Employment issues",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "Employment issues",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues > Funding",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "Funding",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues > Computing occupations",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "Computing occupations",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues > Computing organizations",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "Computing organizations",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues > Testing, certification and licensing",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "Testing, certification and licensing",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing profession > Professional issues > Assistive technologies",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing profession",
    "node_3": "Professional issues",
    "node_4": "Assistive technologies",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Digital rights management",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Digital rights management",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Copyrights",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Copyrights",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Software reverse engineering",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Software reverse engineering",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Patents",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Patents",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Trademarks",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Trademarks",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Internet governance / domain names",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Internet governance / domain names",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Licensing",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Licensing",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Treaties",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Treaties",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Database protection laws",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Database protection laws",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Secondary liability",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Secondary liability",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Soft intellectual property",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Soft intellectual property",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Intellectual property > Hardware reverse engineering",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Intellectual property",
    "node_4": "Hardware reverse engineering",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies > Privacy policies",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "Privacy policies",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies > Censorship",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "Censorship",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies > Pornography",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "Pornography",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies > Hate speech",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "Hate speech",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies > Political speech",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "Political speech",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies > Technology and censorship",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "Technology and censorship",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Content and privacy policies > Censoring filters",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Content and privacy policies",
    "node_4": "Censoring filters",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Surveillance",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Surveillance",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Surveillance > Governmental surveillance",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Surveillance",
    "node_4": "Governmental surveillance",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Surveillance > Corporate surveillance",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Surveillance",
    "node_4": "Corporate surveillance",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy > Commerce policy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "Commerce policy",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy > Taxation",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "Taxation",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy > Transborder data flow",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "Transborder data flow",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy > Antitrust and competition",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "Antitrust and competition",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy > Governmental regulations",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "Governmental regulations",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy > Online auctions policy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "Online auctions policy",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Commerce and regulatory policy > Consumer products policy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Commerce and regulatory policy",
    "node_4": "Consumer products policy",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Network access control",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Network access control",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Censoring filters",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Censoring filters",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Broadband access",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Broadband access",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Net neutrality",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Net neutrality",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Network access restrictions",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Network access restrictions",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Age-based restrictions",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Age-based restrictions",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Acceptable use policy restrictions",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Acceptable use policy restrictions",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Network access policy > Universal access",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Network access policy",
    "node_4": "Universal access",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Computer crime",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Computer crime",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Computer crime > Social engineering attacks",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Computer crime",
    "node_4": "Social engineering attacks",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Computer crime > Spoofing attacks",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Computer crime",
    "node_4": "Spoofing attacks",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Computer crime > Phishing",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Computer crime",
    "node_4": "Phishing",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Computer crime > Identity theft",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Computer crime",
    "node_4": "Identity theft",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Computer crime > Financial crime",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Computer crime",
    "node_4": "Financial crime",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Computer crime > Malware / spyware crime",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Computer crime",
    "node_4": "Malware / spyware crime",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Government technology policy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Government technology policy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Government technology policy > Governmental regulations",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Government technology policy",
    "node_4": "Governmental regulations",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Government technology policy > Import / export controls",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Government technology policy",
    "node_4": "Import / export controls",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy > Medical records",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "Medical records",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy > Personal health records",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "Personal health records",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy > Genetic information",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "Genetic information",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy > Patient privacy",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "Patient privacy",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy > Health information exchanges",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "Health information exchanges",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy > Medical technologies",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "Medical technologies",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > Computing / technology policy > Medical information policy > Remote medicine",
    "high_level_domain": "Social and professional topics",
    "subdomain": "Computing / technology policy",
    "node_3": "Medical information policy",
    "node_4": "Remote medicine",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Race and ethnicity",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Race and ethnicity",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Religious orientation",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Religious orientation",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Gender",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Gender",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Men",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Men",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Women",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Women",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Sexual orientation",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Sexual orientation",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > People with disabilities",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "People with disabilities",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Geographic characteristics",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Geographic characteristics",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Cultural characteristics",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Cultural characteristics",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Age",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Age",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Children",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Children",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Seniors",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Seniors",
    "node_5": ""
  },
  {
    "path": "Social and professional topics > User characteristics > Demographic characteristics > Adolescents",
    "high_level_domain": "Social and professional topics",
    "subdomain": "User characteristics",
    "node_3": "Demographic characteristics",
    "node_4": "Adolescents",
    "node_5": ""
  }
]


acm_json = json.dumps(acm_ccs_structure, indent=4)


In [7]:
import os
import json
import csv
import time
from openai import OpenAI
from dotenv import load_dotenv

# ================== OPENAI CLIENT ==================

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# ================== HELPERS ==================

def save_incremental_results(results, output_file="acm_results_incremental.jsonl"):
    with open(output_file, "a", encoding="utf-8") as file:
        for paper_id, result in results.items():
            file.write(json.dumps({paper_id: result}, ensure_ascii=False) + "\n")


def load_saved_results(output_file="acm_results_incremental.jsonl"):
    saved_ids = set()
    saved_results = {}

    try:
        with open(output_file, "r", encoding="utf-8") as file:
            for line_num, line in enumerate(file, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    result = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Skipping bad JSON line {line_num}")
                    continue

                for paper_id, data in result.items():
                    saved_ids.add(paper_id)
                    saved_results[paper_id] = data

    except FileNotFoundError:
        print("No saved results found. Starting fresh.")

    return saved_ids, saved_results


def load_jsonl(file_path):
    papers = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                papers.append(json.loads(line))
    return papers


def try_parse_json(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()
    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    try:
        return json.loads(text)
    except:
        return text


def dedupe_and_limit(items, max_items=3):
    if not isinstance(items, list):
        return items

    seen = set()
    out = []

    for item in items:
        if not isinstance(item, dict):
            continue

        key = (
            item.get("high_level_domain", ""),
            item.get("subdomain", "")
        )

        if key not in seen:
            seen.add(key)
            out.append(item)

        if len(out) >= max_items:
            break

    return out


# ================== TAXONOMY HELPERS ==================

def build_sub_subdomain_map(flat_taxonomy):
    mapping = {}

    for item in flat_taxonomy:
        high = item.get("high_level_domain", "").strip()
        sub = item.get("subdomain", "").strip()

        if sub:
            mapping[sub.lower()] = {
                "high_level_domain": high,
                "subdomain": sub
            }

        for key in ["node_3", "node_4", "node_5", "node_6"]:
            value = item.get(key, "").strip()
            if value:
                mapping[value.lower()] = {
                    "high_level_domain": high,
                    "subdomain": sub
                }

    return mapping


def build_allowed_structure(flat_taxonomy):
    grouped = {}

    for item in flat_taxonomy:
        high = item.get("high_level_domain", "").strip()
        sub = item.get("subdomain", "").strip()

        if not high or not sub:
            continue

        if high not in grouped:
            grouped[high] = set()

        grouped[high].add(sub)

    allowed_structure = []
    for high, subs in grouped.items():
        allowed_structure.append({
            "high_level_domain": high,
            "subdomains": sorted(subs)
        })

    return allowed_structure


In [8]:
def generate_system_prompt(paper, task, allowed_json, subdomain_map_json):
    title = paper.get("title", "").strip()
    text = paper.get("text", "").strip()

    paper_text = f"{title}\n\n{text}"

    if task == "domain":
 
        return f"""
        You are tasked with identifying the ACM Computing Classification System (CCS) research domains for the following cybersecurity paper titled: "{title}".
        Your job is to return the correct "high_level_domain" and its corresponding "subdomain".
        Important:
        \t- The final output MUST contain ONLY valid "high_level_domain" and "subdomain" values from the allowed ACM CCS structure below.
        \t- The output "subdomain" must be EXACTLY one of the allowed subdomain strings.
        \t- Do NOT output lower-level indicators such as node_3, node_4, node_5 or node_6.
        \t- Lower-level indicators are evidence cues only.
        \t- If the paper mentions a lower-level indicator, you MUST map it back to its parent **subdomain** using the mapping provided below.
        \t- Never invent or paraphrase labels.

        
        Classification rules:
        1. Use ONLY the allowed ACM CCS labels below.
        2. If a paper mentions a lower-level CCS indicator, map it upward to the correct parent subdomain.
        3. If multiple subdomains are clearly supported, you may return multiple pairs.
        4. Return at most 2 pairs in normal cases.
        5. Return 3 pairs only if all 3 are very clearly supported.
        6. Do not over-predict.
        7. You must return at least one pair if the paper contains usable technical content.  
        8. Return an empty list only if the provided text is completely missing or non-informative.
        9. If evidence is weak, return the single closest supported pair with brief evidence.
       
        Venue-specific rules:
        7. For NDSS, USENIX Security, and IEEE S&P:
        \t- First look only at the title, abstract, and introduction.
        \t- These venues are security-centric, so at least one returned pair will usually belong to "Security and privacy" if supported by the paper.
        \t- Your first priority is to identify the correct security subdomain.
        \t- If the paper clearly also belongs to another non-security ACM area, return that too.
        \t- Only check the rest of the body if title, abstract, and introduction are insufficient.
        
        8. Special Rule for ACM CCS:
         \t- If the venue is **ACM CCS**, you MUST determine the domain EXCLUSIVELY from the "CCS Concepts" section that appears on the first page.
         \t- Read the CCS Concepts block and map ONLY the listed CCS terms to the closest (high_level_domain, subdomain) pairs in the ACM CCS structure.
         \t- You MUST NOT add any domains or subdomains that are not explicitly listed in the **CCS Concepts section**.
         \t- Only fall back to scanning the rest of the body IF AND ONLY IF the CCS Concepts section is completely missing.


        9. Json Output Examples, CCS Papers:
         Example #1 : For the paper titled **"Nebula: Efficient, Private and Accurate Histogram Estimation"** map it with the following CCS concepts:

         • Security and privacy → Privacy-preserving protocols

         You should respond with:

         ```json
         [
           {{
             "high_level_domain": "Security and privacy",
             "subdomain": "Security services"
             "evidence": ["No Evidence Needed, CCS Concept Exist"]
           }}
         ]
         ```
         For NON-CSS Paper (NDSS, USENIX Security, and IEEE S&P)
         Example #2: For the paper For the paper titled **"28 Blinks Later Tackling Practical Challenges of Eye Movement Biometrics."**
         ```json
         [
           {{
             "high_level_domain": "Security and privacy",
             "subdomain": "Security services",
             "evidence": ["Eye Movement Biometrics", "biometrics"]
           }},
           {{
             "high_level_domain": "Computing methodologies",
             "subdomain": "Artificial intelligence",
             "evidence": ["Eye Movement Biometrics", "biometrics"]
           }}
         ]
         ```

        
        Allowed ACM CCS structure:
        <ACM_CCS_START>
        {allowed_json}
        <ACM_CCS_END>
        Lower-level indicator to parent-subdomain mapping:
        <SUB_SUBDOMAIN_MAPPING>
        {subdomain_map_json}
        </SUB_SUBDOMAIN_MAPPING>
        Return JSON only in this format:
        
       Output JSON only:
       [
        {{
          "high_level_domain": "...",
          "subdomain": "...",
          "evidence": ["matched phrase 1", "matched phrase 2"]
        }}
        ]

     Paper:
     {paper_text}
     
     """
    else:
        return f"Error: Task '{task}' is not supported"

In [ ]:
# ================== MAIN PROCESS ==================

def process_papers_for_domain_task(
    papers,
    flat_taxonomy,
    start_index=0,
    output_file="domain_results_incremental.jsonl"
):
    task_results = {}

    allowed_structure = build_allowed_structure(flat_taxonomy)
    subdomain_map = build_sub_subdomain_map(flat_taxonomy)

    allowed_json = json.dumps(allowed_structure, ensure_ascii=False)
    subdomain_map_json = json.dumps(subdomain_map, ensure_ascii=False)

    total_count = len(papers) + start_index

    for i, paper in enumerate(papers, start=start_index):
        paper_id = paper["paper_id"]
        title = paper.get("title", "")

        print(f"\nProcessing {i+1}/{total_count}")
        print(f"ID: {paper_id}")
        print(f"Title: {title}")

        if paper_id in processed_ids:
            print("Skipping (already done)")
            continue

        task_results[paper_id] = {
            "paper_id": paper_id,
            "title": title
        }

        try:
            prompt = generate_system_prompt(
                paper,
                "domain",
                allowed_json,
                subdomain_map_json
            )

            response = client.chat.completions.create(
                model="gpt-5-mini",
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=5000
            )

            raw = response.choices[0].message.content
            parsed = try_parse_json(raw)
            cleaned = dedupe_and_limit(parsed)

            print("Result:", raw)
            task_results[paper_id]["domain"] = cleaned

        except Exception as e:
            print("Error:", e)
            task_results[paper_id]["domain"] = f"error: {e}"
            time.sleep(2)

        save_incremental_results({paper_id: task_results[paper_id]}, output_file)

    return task_results


# ================== LOAD ==================

output_file = "domain_results_incremental.jsonl"
processed_ids, processed_results = load_saved_results(output_file)

input_file = "new_pipeline_all_dataset_papers.jsonl"
all_papers = load_jsonl(input_file)

remaining_papers = [
    p for p in all_papers
    if p["paper_id"] not in processed_ids
]

# ================== RUN ==================

all_results = process_papers_for_domain_task(
    remaining_papers,
    flat_taxonomy=acm_ccs_structure,
    start_index=len(processed_ids),
    output_file=output_file
)

processed_results.update(all_results)

# ================== SAVE FINAL ==================

with open("domain_results_final.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["paper_id", "title", "domain"])

    for pid, res in processed_results.items():
        domain = res.get("domain", "")
        if isinstance(domain, (dict, list)):
            domain = json.dumps(domain, ensure_ascii=False)

        writer.writerow([pid, res.get("title", ""), domain])

with open("domain_results_final.jsonl", "w", encoding="utf-8") as f:
    for pid, res in processed_results.items():
        f.write(json.dumps({pid: res}, ensure_ascii=False) + "\n")

print("Done.")